# SoleFormer: 3D Pose Estimation from Insole Sensors - Complete Pipeline

This notebook implements the complete SoleFormer pipeline:
1. **Data Preprocessing**: Load, normalize, and prepare pressure and IMU data
2. **Feature Extraction**: Graph Neural Network for pressure, MLP for IMU
3. **Two-Stream Transformer**: Self-attention + Cross-attention architecture
4. **Double-Cycle Consistency Loss**: Enforce physical constraints via AccelNet & PressNet
5. **Training & Evaluation**: End-to-end training with validation

Based on: SolePoser: Real-Time 3D Human Pose Estimation using Insole Pressure Sensors (Wu et al., UIST 2024)

In [4]:
# Import libraries
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.signal import savgol_filter
import seaborn as sns
from tqdm import tqdm
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

def _find_project_root_for_import(start_dir=None):
    if start_dir is None:
        start_dir = Path.cwd()
    start_dir = Path(start_dir).resolve()
    for p in [start_dir] + list(start_dir.parents):
        if (p / 'notebooks' / 'loader.py').exists():
            return p
    return start_dir

# Ensure project root is importable in notebook kernels launched from subfolders
PROJECT_ROOT = _find_project_root_for_import()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Try importing project loader utility for insole channel restructuring
try:
    from notebooks.loader import restructure_insole_data
    HAS_PROJECT_LOADER = True
except Exception as e:
    HAS_PROJECT_LOADER = False
    print(f"Project loader import warning: {e}")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Project loader available: {HAS_PROJECT_LOADER}")

PyTorch version: 2.7.0
CUDA available: False
Device: cpu
Project root: C:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel
Project loader available: True


## 1. Data Preprocessing

### Input Data Format:
- **Pressure Data**: 16-channel foot pressure sensors (0-20 N/cm²)
- **IMU Data**: 6DoF (3-axis acceleration + 3-axis gyroscope)
- **Target**: 3D joint positions (17 joints × 3 dimensions = 51 output features)

### Preprocessing Steps:
1. Load data from CSV files
2. Normalize pressure to [0, 1] range
3. Center and normalize joint positions
4. Apply temporal smoothing
5. Create sequences for temporal learning

In [7]:
class PressureIMUDataPreprocessor:
    """Preprocessor for insole sensor and pose data."""
    
    def __init__(self, pressure_clip_max=20.0, smoothing_sigma=1.0):
        """Initialize preprocessor.
        
        Args:
            pressure_clip_max: Maximum pressure value in N/cm² (default: 20)
            smoothing_sigma: Sigma for Gaussian smoothing
        """
        self.pressure_clip_max = pressure_clip_max
        self.smoothing_sigma = smoothing_sigma
        self.pressure_scaler = MinMaxScaler()
        self.imu_scaler = StandardScaler()
        self.pose_scaler = StandardScaler()
    
    def preprocess_pressure(self, pressure_data):
        """Normalize pressure data to [0, 1].
        
        Args:
            pressure_data: (n_samples, 32) - 16 sensors × 2 feet
        
        Returns:
            normalized: (n_samples, 32)
        """
        # Clip to valid range
        clipped = np.clip(pressure_data, 0, self.pressure_clip_max)
        # Scale to [0, 1]
        normalized = self.pressure_scaler.fit_transform(clipped)
        return normalized
    
    def preprocess_imu(self, imu_data):
        """Normalize IMU data (6DoF acceleration + gyroscope).
        
        Args:
            imu_data: (n_samples, 12) - 6DoF × 2 feet
        
        Returns:
            normalized: (n_samples, 12)
        """
        # Standardize to zero mean and unit variance
        normalized = self.imu_scaler.fit_transform(imu_data)
        return normalized
    
    def preprocess_pose(self, pose_data):
        """Normalize pose data.
        
        Args:
            pose_data: (n_samples, n_joints*3) - 3D joint positions
        
        Returns:
            normalized: (n_samples, n_joints*3)
        """
        # Center by hip position (joint 0)
        n_joints = pose_data.shape[1] // 3
        pose_centered = pose_data.copy()
        hip_x, hip_y, hip_z = pose_data[:, 0:3].mean(axis=0)
        for i in range(n_joints):
            pose_centered[:, i*3:i*3+3] -= np.array([hip_x, hip_y, hip_z])
        
        # Standardize
        normalized = self.pose_scaler.fit_transform(pose_centered)
        return normalized
    
    def apply_temporal_smoothing(self, data, sigma=None):
        """Apply Gaussian smoothing along time axis.
        
        Args:
            data: (n_samples, n_features)
            sigma: Smoothing sigma (uses self.smoothing_sigma if None)
        
        Returns:
            smoothed: (n_samples, n_features)
        """
        if sigma is None:
            sigma = self.smoothing_sigma
        
        if sigma <= 0:
            return data
        
        from scipy.ndimage import gaussian_filter1d
        smoothed = np.zeros_like(data)
        for i in range(data.shape[1]):
            smoothed[:, i] = gaussian_filter1d(data[:, i], sigma=sigma)
        return smoothed

print("Data Preprocessor defined.")

Data Preprocessor defined.


## 2. Synthetic Data Generation

For demonstration purposes, we'll generate synthetic data. In practice, you would load real data from the SolePose dataset.

In [22]:
def generate_synthetic_data(n_samples=1000, n_joints=17, random_seed=42):
    """Generate synthetic pressure, IMU, and pose data (fallback mode)."""
    np.random.seed(random_seed)
    t = np.arange(n_samples) / n_samples * 2 * np.pi

    pressure = np.zeros((n_samples, 32), dtype=np.float32)
    for i in range(16):
        pressure[:, i] = 10 * (np.sin(t + i / 8) + 1) / 2
        pressure[:, 16 + i] = 10 * (np.sin(t + i / 8 + np.pi) + 1) / 2
    pressure += np.random.normal(0, 0.5, pressure.shape).astype(np.float32)
    pressure = np.clip(pressure, 0, 20)

    imu = np.zeros((n_samples, 12), dtype=np.float32)
    for i in range(6):
        imu[:, i] = 5 * np.sin(t * 2 + i)
        imu[:, 6 + i] = 5 * np.sin(t * 2 + i + np.pi)
    imu += np.random.normal(0, 0.3, imu.shape).astype(np.float32)

    pose = np.zeros((n_samples, n_joints * 3), dtype=np.float32)
    for j in range(n_joints):
        pose[:, j * 3] = 0.5 * np.sin(t + j / n_joints) + np.random.normal(0, 0.1, n_samples)
        pose[:, j * 3 + 1] = 0.5 * np.cos(t + j / n_joints) + np.random.normal(0, 0.1, n_samples)
        pose[:, j * 3 + 2] = j * 0.1 + 0.2 * np.sin(t) + np.random.normal(0, 0.1, n_samples)

    return pressure.astype(np.float32), imu.astype(np.float32), pose.astype(np.float32)


def _read_insole_auto(file_path):
    df = pd.read_csv(file_path, engine='python')
    if df.shape[1] == 1:
        df = pd.read_csv(file_path, sep='\t', engine='python')
    if df.shape[1] == 1:
        df = pd.read_csv(file_path, sep=';', engine='python')
    df.columns = df.columns.str.strip()
    return df


def _tag_from_name(path_name):
    name = Path(path_name).stem
    if name.startswith('Awinda_'):
        return name[len('Awinda_'):]
    if name.startswith('Soles_'):
        return name[len('Soles_'):]
    return None


def _subject_key_from_tag(tag):
    """Extract subject key from tag like '001CcSs_2_tech_paos'."""
    if not isinstance(tag, str) or len(tag) == 0:
        return None
    return tag.split('_')[0]


def _pair_top_level_files(skeleton_dir, insole_dir):
    """Pair top-level files only (non-recursive), ignoring subfolders."""
    skeleton_dir = Path(skeleton_dir)
    insole_dir = Path(insole_dir)
    if not skeleton_dir.exists():
        raise FileNotFoundError(f"Skeleton dir not found: {skeleton_dir}")
    if not insole_dir.exists():
        raise FileNotFoundError(f"Insole dir not found: {insole_dir}")

    # TOP-LEVEL ONLY: do not recurse into subfolders
    skeleton_files = sorted([p for p in skeleton_dir.glob('Awinda_*.csv') if p.is_file()])
    insole_files = sorted([p for p in insole_dir.glob('Soles_*.txt') if p.is_file()])

    sk_map = {_tag_from_name(p.name): p for p in skeleton_files}
    in_map = {_tag_from_name(p.name): p for p in insole_files}

    common_tags = sorted(set(sk_map.keys()) & set(in_map.keys()))
    if not common_tags:
        raise ValueError(
            f"No matching Awinda_/Soles_ tags between {skeleton_dir} and {insole_dir}."
        )

    missing_sk = sorted(set(in_map.keys()) - set(sk_map.keys()))
    missing_in = sorted(set(sk_map.keys()) - set(in_map.keys()))
    if missing_sk:
        print(f"Warning: {len(missing_sk)} insole files have no matching skeleton file.")
    if missing_in:
        print(f"Warning: {len(missing_in)} skeleton files have no matching insole file.")

    return [(tag, sk_map[tag], in_map[tag]) for tag in common_tags]


def _load_split_tolerant(skeleton_dir, insole_dir, split_name='split'):
    pairs = _pair_top_level_files(skeleton_dir, insole_dir)
    all_skeleton, all_insole, all_segment_ids, all_subject_keys = [], [], [], []
    trim_report = []

    for segment_id, (tag, sk_path, in_path) in enumerate(pairs, start=1):
        sk_df = pd.read_csv(sk_path)
        sk_df = sk_df.drop(columns=['Frame', 'frame', '# time', 'time', 'Timestamp'], errors='ignore')
        in_df = _read_insole_auto(in_path)

        n_sk, n_in = len(sk_df), len(in_df)
        n_min = min(n_sk, n_in)
        if n_min == 0:
            continue
        if n_sk != n_in:
            trim_report.append((tag, n_sk, n_in, n_min))

        sk_df = sk_df.iloc[:n_min].reset_index(drop=True)
        in_df = in_df.iloc[:n_min].reset_index(drop=True)

        all_skeleton.append(sk_df)
        all_insole.append(in_df)
        all_segment_ids.append(np.full(n_min, segment_id, dtype=np.int32))
        subj_key = _subject_key_from_tag(tag)
        all_subject_keys.append(np.full(n_min, subj_key, dtype=object))

    if not all_skeleton:
        raise ValueError(f"No valid paired files were loaded for {split_name}.")

    skeleton_df = pd.concat(all_skeleton, ignore_index=True)
    insole_df = pd.concat(all_insole, ignore_index=True)
    segment_ids = np.concatenate(all_segment_ids, axis=0)
    subject_keys = np.concatenate(all_subject_keys, axis=0)

    if trim_report:
        print(f"[{split_name}] Trimmed {len(trim_report)} pair(s) due to frame mismatch.")
        for tag, n_sk, n_in, n_min in trim_report[:5]:
            print(f"  - {tag}: skeleton={n_sk}, insole={n_in}, using={n_min}")
        if len(trim_report) > 5:
            print(f"  ... and {len(trim_report) - 5} more")

    return skeleton_df, insole_df, segment_ids, subject_keys


def load_subject_anthropometrics(project_root=None, file_name='subject_anthropo.txt'):
    """Load subject anthropometric table from data folder."""
    if project_root is None:
        project_root = PROJECT_ROOT
    path = Path(project_root).resolve() / 'data' / file_name
    if not path.exists():
        raise FileNotFoundError(f"Anthropometric file not found: {path}")

    df = pd.read_csv(path, sep=None, engine='python')
    df.columns = [str(c).strip() for c in df.columns]
    if 'subject_key' not in df.columns:
        raise ValueError(f"Anthropometric file must contain a 'subject_key' column: {path}")
    df['subject_key'] = df['subject_key'].astype(str).str.strip()
    return df


def load_train_test_from_structure(project_root=None):
    if project_root is None:
        project_root = PROJECT_ROOT
    project_root = Path(project_root).resolve()

    train_sk_dir = project_root / 'data' / 'training_data' / 'skeleton'
    train_in_dir = project_root / 'data' / 'training_data' / 'Insole'
    test_sk_dir = project_root / 'data' / 'test_data' / 'skeleton'
    test_in_dir = project_root / 'data' / 'test_data' / 'Insole'

    train_skeleton_df, train_insole_df, train_segment_ids, train_subject_keys = _load_split_tolerant(
        train_sk_dir, train_in_dir, split_name='train'
    )
    test_skeleton_df, test_insole_df, test_segment_ids, test_subject_keys = _load_split_tolerant(
        test_sk_dir, test_in_dir, split_name='test'
    )

    # Numeric cleanup
    train_skeleton_df = train_skeleton_df.apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all').ffill().bfill().fillna(0.0)
    test_skeleton_df = test_skeleton_df.apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all').ffill().bfill().fillna(0.0)

    # Align target schemas between train/test
    common_cols = [c for c in train_skeleton_df.columns if c in test_skeleton_df.columns]
    if not common_cols:
        raise ValueError('No common target columns between training and testing skeleton files.')
    train_skeleton_df = train_skeleton_df[common_cols]
    test_skeleton_df = test_skeleton_df[common_cols]

    # Insole split into pressure + IMU
    if HAS_PROJECT_LOADER:
        train_pressure_df, train_imu_df = restructure_insole_data(train_insole_df)
        test_pressure_df, test_imu_df = restructure_insole_data(test_insole_df)
    else:
        raise RuntimeError('restructure_insole_data is unavailable; cannot parse insole channels.')

    train_pressure_df = train_pressure_df.apply(pd.to_numeric, errors='coerce').fillna(0.0)
    train_imu_df = train_imu_df.apply(pd.to_numeric, errors='coerce').fillna(0.0)
    test_pressure_df = test_pressure_df.apply(pd.to_numeric, errors='coerce').fillna(0.0)
    test_imu_df = test_imu_df.apply(pd.to_numeric, errors='coerce').fillna(0.0)

    return (
        train_pressure_df.to_numpy(dtype=np.float32),
        train_imu_df.to_numpy(dtype=np.float32),
        train_skeleton_df.to_numpy(dtype=np.float32),
        train_segment_ids,
        train_subject_keys,
        test_pressure_df.to_numpy(dtype=np.float32),
        test_imu_df.to_numpy(dtype=np.float32),
        test_skeleton_df.to_numpy(dtype=np.float32),
        test_segment_ids,
        test_subject_keys,
    )


def build_anthropometric_feature_matrices(train_subject_keys, test_subject_keys, project_root=None):
    """Create frame-level anthropometric features aligned with train/test rows."""
    anthro_df = load_subject_anthropometrics(project_root=project_root)

    feature_cols = [c for c in anthro_df.columns if c != 'subject_key']
    if not feature_cols:
        raise ValueError('No anthropometric feature columns found (expected columns besides subject_key).')

    for col in feature_cols:
        anthro_df[col] = pd.to_numeric(anthro_df[col], errors='coerce')

    feature_means = anthro_df[feature_cols].mean(axis=0)
    anthro_df[feature_cols] = anthro_df[feature_cols].fillna(feature_means)

    anthro_map = anthro_df.set_index('subject_key')[feature_cols]

    def _subject_keys_to_matrix(keys):
        rows = []
        missing_keys = set()
        for key in keys:
            key = str(key).strip()
            if key in anthro_map.index:
                rows.append(anthro_map.loc[key].to_numpy(dtype=np.float32))
            else:
                rows.append(feature_means.to_numpy(dtype=np.float32))
                missing_keys.add(key)
        mat = np.asarray(rows, dtype=np.float32)
        return mat, sorted(missing_keys)

    train_mat, missing_train = _subject_keys_to_matrix(train_subject_keys)
    test_mat, missing_test = _subject_keys_to_matrix(test_subject_keys)
    missing_all = sorted(set(missing_train + missing_test))
    if missing_all:
        print(f"Warning: missing anthropometric rows for keys: {missing_all}. Using dataset mean values.")

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_mat).astype(np.float32)
    test_scaled = scaler.transform(test_mat).astype(np.float32)

    return train_scaled, test_scaled, feature_cols, scaler


############################
# DATA SOURCE SWITCH
############################
USE_REAL_DATA = True  # True: training_data/test_data folders, False: synthetic fallback

print('Preparing dataset...')
DATA_SOURCE = 'synthetic'
DATA_LOAD_ERROR = None

if USE_REAL_DATA:
    try:
        (
            pressure_train_raw, imu_train_raw, pose_train_raw, train_segment_ids, train_subject_keys,
            pressure_test_raw, imu_test_raw, pose_test_raw, test_segment_ids, test_subject_keys,
        ) = load_train_test_from_structure(PROJECT_ROOT)
        DATA_SOURCE = 'real'
        print('Loaded REAL train/test splits from training_data and test_data folders (top-level files only).')
    except Exception as e:
        DATA_SOURCE = 'synthetic-fallback'
        DATA_LOAD_ERROR = str(e)
        print(f'Real-data loading failed: {DATA_LOAD_ERROR}')
        print('Falling back to synthetic data.')
        pressure_train_raw, imu_train_raw, pose_train_raw = generate_synthetic_data(n_samples=2000, n_joints=17)
        pressure_test_raw, imu_test_raw, pose_test_raw = generate_synthetic_data(n_samples=800, n_joints=17, random_seed=123)
        train_segment_ids = None
        test_segment_ids = None
        train_subject_keys = None
        test_subject_keys = None
else:
    pressure_train_raw, imu_train_raw, pose_train_raw = generate_synthetic_data(n_samples=2000, n_joints=17)
    pressure_test_raw, imu_test_raw, pose_test_raw = generate_synthetic_data(n_samples=800, n_joints=17, random_seed=123)
    train_segment_ids = None
    test_segment_ids = None
    train_subject_keys = None
    test_subject_keys = None

print(f'Data source: {DATA_SOURCE}')
print(f'Train shapes -> pressure: {pressure_train_raw.shape}, imu: {imu_train_raw.shape}, pose: {pose_train_raw.shape}')
print(f'Test shapes  -> pressure: {pressure_test_raw.shape}, imu: {imu_test_raw.shape}, pose: {pose_test_raw.shape}')
if train_subject_keys is not None:
    print(f'Train subject keys: {len(np.unique(train_subject_keys))} unique')
if test_subject_keys is not None:
    print(f'Test subject keys: {len(np.unique(test_subject_keys))} unique')

Preparing dataset...
Loaded REAL train/test splits from training_data and test_data folders (top-level files only).
Data source: real
Train shapes -> pressure: (226353, 32), imu: (226353, 12), pose: (226353, 69)
Test shapes  -> pressure: (10846, 32), imu: (10846, 12), pose: (10846, 69)
Train subject keys: 5 unique
Test subject keys: 1 unique


In [ ]:
# Compact diagnostics for loaded train/test datasets
print('--- Train/Test Data Diagnostics ---')
print(f'DATA_SOURCE: {DATA_SOURCE if "DATA_SOURCE" in globals() else "unknown"}')
if 'DATA_LOAD_ERROR' in globals() and DATA_LOAD_ERROR:
    print(f'DATA_LOAD_ERROR: {DATA_LOAD_ERROR}')

print(f'train pressure/imu/pose: {pressure_train_raw.shape} / {imu_train_raw.shape} / {pose_train_raw.shape}')
print(f'test  pressure/imu/pose: {pressure_test_raw.shape} / {imu_test_raw.shape} / {pose_test_raw.shape}')
print(f'dtypes train: {pressure_train_raw.dtype}, {imu_train_raw.dtype}, {pose_train_raw.dtype}')
print(f'dtypes test : {pressure_test_raw.dtype}, {imu_test_raw.dtype}, {pose_test_raw.dtype}')

if 'train_segment_ids' in globals() and train_segment_ids is not None:
    _, train_counts = np.unique(train_segment_ids, return_counts=True)
    print(f'train segments: {len(np.unique(train_segment_ids))}, min/median/max frames: {train_counts.min()}/{int(np.median(train_counts))}/{train_counts.max()}')
else:
    print('train_segment_ids: None')

if 'test_segment_ids' in globals() and test_segment_ids is not None:
    _, test_counts = np.unique(test_segment_ids, return_counts=True)
    print(f'test segments: {len(np.unique(test_segment_ids))}, min/median/max frames: {test_counts.min()}/{int(np.median(test_counts))}/{test_counts.max()}')
else:
    print('test_segment_ids: None')

--- Train/Test Data Diagnostics ---
DATA_SOURCE: real
train pressure/imu/pose: (226353, 32) / (226353, 12) / (226353, 69)
test  pressure/imu/pose: (10846, 32) / (10846, 12) / (10846, 69)
dtypes train: float32, float32, float32
dtypes test : float32, float32, float32
train segments: 19, min/median/max frames: 9661/11252/21033
test segments: 1, min/median/max frames: 10846/10846/10846


In [28]:
# Preprocess data: fit on TRAIN only, transform TEST with train-fitted scalers
print('\nPreprocessing train/test data...')
preprocessor = PressureIMUDataPreprocessor(pressure_clip_max=20.0, smoothing_sigma=1.0)

# Pressure: clip then fit/train transform + test transform
pressure_train_clip = np.clip(pressure_train_raw, 0, preprocessor.pressure_clip_max)
pressure_test_clip = np.clip(pressure_test_raw, 0, preprocessor.pressure_clip_max)
pressure_train_processed = preprocessor.pressure_scaler.fit_transform(pressure_train_clip)
pressure_test_processed = preprocessor.pressure_scaler.transform(pressure_test_clip)

# IMU: fit on train, transform test
imu_train_processed = preprocessor.imu_scaler.fit_transform(imu_train_raw)
imu_test_processed = preprocessor.imu_scaler.transform(imu_test_raw)

# Pose: center test by TRAIN hip mean for consistent frame, then standardize using train stats
n_joints_tmp = pose_train_raw.shape[1] // 3
train_hip_mean = pose_train_raw[:, 0:3].mean(axis=0)
pose_train_centered = pose_train_raw.copy()
pose_test_centered = pose_test_raw.copy()
for i in range(n_joints_tmp):
    pose_train_centered[:, i*3:i*3+3] -= train_hip_mean
    pose_test_centered[:, i*3:i*3+3] -= train_hip_mean

pose_train_processed = preprocessor.pose_scaler.fit_transform(pose_train_centered)
pose_test_processed = preprocessor.pose_scaler.transform(pose_test_centered)

# EXPLICIT LEARNING: load anthropometrics for constraints/post-processing (not model input)
anthropo_feature_names = []
anthropo_scaler = None
anthropo_train_raw = None
anthropo_test_raw = None
if (
    DATA_SOURCE == 'real'
    and train_subject_keys is not None
    and test_subject_keys is not None
):
    try:
        (
            anthropo_train_scaled,
            anthropo_test_scaled,
            anthropo_feature_names,
            anthropo_scaler,
        ) = build_anthropometric_feature_matrices(
            train_subject_keys=train_subject_keys,
            test_subject_keys=test_subject_keys,
            project_root=PROJECT_ROOT,
        )

        # Use DIRECT subject anthropometrics (original units) for explicit constraints.
        if anthropo_scaler is not None:
            anthropo_train_raw = anthropo_scaler.inverse_transform(anthropo_train_scaled).astype(np.float32)
            anthropo_test_raw = anthropo_scaler.inverse_transform(anthropo_test_scaled).astype(np.float32)
        else:
            anthropo_train_raw = anthropo_train_scaled.astype(np.float32)
            anthropo_test_raw = anthropo_test_scaled.astype(np.float32)

        print(f"Anthropometric features loaded (direct values for explicit constraints): {len(anthropo_feature_names)} -> {anthropo_feature_names}")
    except Exception as e:
        print(f"Anthropometric loading warning: {e}. Continuing without explicit anthropometric constraints.")
        anthropo_train_raw = None
        anthropo_test_raw = None
        anthropo_feature_names = []
        anthropo_scaler = None

# Keep empty arrays for model input compatibility (anthropo is NOT concatenated to input)
anthropo_train_processed = np.zeros((len(pressure_train_processed), 0), dtype=np.float32)
anthropo_test_processed = np.zeros((len(pressure_test_processed), 0), dtype=np.float32)

print(f'Train processed pressure range: [{pressure_train_processed.min():.3f}, {pressure_train_processed.max():.3f}]')
print(f'Test processed pressure range:  [{pressure_test_processed.min():.3f}, {pressure_test_processed.max():.3f}]')
print(f'Train processed IMU mean/std: {imu_train_processed.mean():.3f}/{imu_train_processed.std():.3f}')
print(f'Test processed IMU mean/std:  {imu_test_processed.mean():.3f}/{imu_test_processed.std():.3f}')
print(f'Train processed pose mean/std: {pose_train_processed.mean():.3f}/{pose_train_processed.std():.3f}')
print(f'Test processed pose mean/std:  {pose_test_processed.mean():.3f}/{pose_test_processed.std():.3f}')
if anthropo_train_raw is not None:
    print(f'Anthropometric raw shapes -> train: {anthropo_train_raw.shape}, test: {anthropo_test_raw.shape}')
else:
    print('Anthropometric raw shapes -> train/test: None')


Preprocessing train/test data...
Anthropometric features loaded (direct values for explicit constraints): 11 -> ['height', 'foot_length', 'shoulder_height', 'shoulder_width', 'elbow_span', 'wrist_span', 'arm_span', 'hip_height', 'hip_width', 'knee_height', 'ankle_height']
Train processed pressure range: [0.000, 1.000]
Test processed pressure range:  [0.000, 1.000]
Train processed IMU mean/std: 0.000/1.000
Test processed IMU mean/std:  0.011/1.036
Train processed pose mean/std: -0.000/1.000
Test processed pose mean/std:  0.468/0.824
Anthropometric raw shapes -> train: (226353, 11), test: (10846, 11)


## 3. Dataset Creation for Sequence Learning

In [ ]:
class PressureIMUPoseSequenceDataset(Dataset):
    """Dataset for sequence-to-sequence pose estimation with segment-safe windows.

    EXPLICIT LEARNING MODE:
    - Model input contains ONLY pressure + IMU.
    - Subject anthropometrics are carried in each sample for explicit constraints.
    """

    def __init__(self, pressure, imu, pose, sequence_length=16, stride=1, segment_ids=None, anthropo=None):
        self.pressure = torch.FloatTensor(pressure)
        self.imu = torch.FloatTensor(imu)
        self.pose = torch.FloatTensor(pose)
        self.sequence_length = int(sequence_length)
        self.stride = int(stride)
        self.segment_ids = None if segment_ids is None else np.asarray(segment_ids)

        self.has_anthropo = anthropo is not None and np.asarray(anthropo).shape[1] > 0
        self.anthropo = None
        if self.has_anthropo:
            self.anthropo = torch.FloatTensor(anthropo)

        if len(self.pressure) != len(self.imu) or len(self.imu) != len(self.pose):
            raise ValueError('pressure, imu, and pose must have identical frame counts')
        if self.segment_ids is not None and len(self.segment_ids) != len(self.pose):
            raise ValueError('segment_ids length must match frame count')
        if self.has_anthropo and len(self.anthropo) != len(self.pose):
            raise ValueError('anthropo length must match frame count')
        if self.sequence_length <= 0:
            raise ValueError('sequence_length must be > 0')

        self.indices = []
        max_start = len(self.pressure) - self.sequence_length
        for i in range(0, max_start + 1, self.stride):
            if self.segment_ids is not None and self.segment_ids[i] != self.segment_ids[i + self.sequence_length - 1]:
                continue
            self.indices.append(i)

        if len(self.indices) == 0:
            raise ValueError('No valid sequences found. Decrease sequence_length or check segment boundaries.')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start_idx = self.indices[idx]
        end_idx = start_idx + self.sequence_length
        pressure_seq = self.pressure[start_idx:end_idx]
        imu_seq = self.imu[start_idx:end_idx]
        pose_seq = self.pose[start_idx:end_idx]

        # Model input: ONLY pressure + IMU
        input_seq = torch.cat([pressure_seq, imu_seq], dim=-1)

        # Keep anthropometrics aligned with sample; collapse temporal window to one vector
        if self.has_anthropo:
            anthropo_seq = self.anthropo[start_idx:end_idx]
            anthropo_vec = anthropo_seq.mean(dim=0)
        else:
            anthropo_vec = torch.empty((0,), dtype=self.pressure.dtype)

        return {
            'input': input_seq,
            'pressure': pressure_seq,
            'imu': imu_seq,
            'anthropo': anthropo_vec,
            'pose': pose_seq,
        }

# Create train/test sequence datasets (no random split)
sequence_length = 64
train_stride = 2
test_stride = 1

train_dataset = PressureIMUPoseSequenceDataset(
    pressure_train_processed,
    imu_train_processed,
    pose_train_processed,
    sequence_length=sequence_length,
    stride=train_stride,
    segment_ids=train_segment_ids if 'train_segment_ids' in globals() else None,
    anthropo=anthropo_train_raw if 'anthropo_train_raw' in globals() and anthropo_train_raw is not None else None,
)

test_dataset = PressureIMUPoseSequenceDataset(
    pressure_test_processed,
    imu_test_processed,
    pose_test_processed,
    sequence_length=sequence_length,
    stride=test_stride,
    segment_ids=test_segment_ids if 'test_segment_ids' in globals() else None,
    anthropo=anthropo_test_raw if 'anthropo_test_raw' in globals() and anthropo_test_raw is not None else None,
)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')
print(f'Sequence length: {sequence_length}, train_stride: {train_stride}, test_stride: {test_stride}')
sample = train_dataset[0]
print(f'Sample input shape (pressure + IMU only): {sample["input"].shape}')
print(f'Sample pose shape: {sample["pose"].shape}')
print(f'Sample anthropo vector shape: {sample["anthropo"].shape}')

Train dataset size: 113035
Test dataset size: 10831
Sequence length: 16, train_stride: 2, test_stride: 1
Sample input shape (pressure + IMU only): torch.Size([16, 44])
Sample pose shape: torch.Size([16, 69])
Sample anthropo vector shape: torch.Size([11])


In [ ]:
# Build dataloaders directly from explicit train/test datasets
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Keep legacy variable name for downstream cells
val_loader = test_loader
val_dataset = test_dataset

print(f'Train dataset size: {len(train_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')
print(f'Train batches: {len(train_loader)}')
print(f'Test batches: {len(test_loader)}')

Train dataset size: 113035
Test dataset size: 10831
Train batches: 3533
Test batches: 339


## 4. Neural Network Architecture

### Components:
1. **PositionalEncoding**: Temporal position information
2. **GraphPressureNet**: Extract spatial relationships from 16 pressure sensors
3. **SoleFormer**: Two-stream transformer with self+cross-attention
4. **AccelNet**: Auxiliary network for IMU cycle consistency
5. **PressNet**: Auxiliary network for pressure cycle consistency

In [ ]:
import math

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for transformer."""
    
    def __init__(self, d_model, max_len=512):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """Add positional encoding to input.
        
        Args:
            x: (batch, seq_len, d_model)
        
        Returns:
            encoded: (batch, seq_len, d_model)
        """
        return x + self.pe[:, :x.size(1)]

print("PositionalEncoding defined.")

PositionalEncoding defined.


In [ ]:
class GraphPressureNet(nn.Module):
    """Graph Neural Network for pressure sensor feature extraction."""
    
    def __init__(self, pressure_dim, d_model, dropout=0.1):
        super().__init__()
        self.num_sensors = 16
        self.pressure_dim = pressure_dim
        
        # Project each sensor pair (left, right) to d_model dimensions
        self.sensor_projection = nn.Sequential(
            nn.Linear(2, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        
        # Graph attention layers (sensors as nodes)
        self.graph_attention = nn.MultiheadAttention(
            d_model, num_heads=4, dropout=dropout, batch_first=True
        )
        
        # Output projection
        self.output_projection = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
        )
    
    def forward(self, x):
        """Extract spatial relationships from pressure data.
        
        Args:
            x: (batch, seq_len, 32) - 16 sensors × 2 feet
        
        Returns:
            output: (batch, seq_len, d_model)
        """
        batch_size, seq_len, _ = x.shape
        
        # Reshape to (batch, seq_len, 16, 2)
        x = x.view(batch_size, seq_len, self.num_sensors, 2)
        
        # Project each sensor
        x = self.sensor_projection(x)  # (batch, seq_len, 16, d_model)
        
        # Flatten for attention
        x_flat = x.view(batch_size * seq_len, self.num_sensors, -1)
        
        # Apply graph attention (self-attention across sensors)
        x_attn, _ = self.graph_attention(x_flat, x_flat, x_flat)
        
        # Project output
        x_out = self.output_projection(x_attn)  # (batch*seq_len, 16, d_model)
        
        # Mean pooling across sensors
        x_pooled = x_out.mean(dim=1)  # (batch*seq_len, d_model)
        
        # Reshape back
        return x_pooled.view(batch_size, seq_len, -1)

print("GraphPressureNet defined.")

GraphPressureNet defined.


In [ ]:
class SoleFormer(nn.Module):
    """Two-stream Transformer with cross-attention for pose estimation.
    
    Architecture:
    - Pressure stream: Graph Neural Network
    - IMU stream: MLP
    - Cross-attention: Learn pressure-IMU relationships
    - Fusion: Concatenate and decode to pose
    """
    
    def __init__(
        self,
        pressure_dim=32,
        imu_dim=12,
        d_model=128,
        nhead=8,
        num_encoder_layers=2,
        output_dim=51,  # 17 joints × 3
        dropout=0.1,
        use_graph_pressure=True,
    ):
        super().__init__()
        
        self.pressure_dim = int(pressure_dim)
        self.imu_dim = int(imu_dim)
        self.output_dim = int(output_dim)
        self.num_joints = self.output_dim // 3
        self.d_model = d_model
        self.num_encoder_layers = num_encoder_layers
        
        # Stream 1: Pressure feature extraction
        if use_graph_pressure:
            self.pressure_feature_extractor = GraphPressureNet(pressure_dim, d_model, dropout)
        else:
            self.pressure_feature_extractor = nn.Sequential(
                nn.Linear(pressure_dim, d_model),
                nn.LayerNorm(d_model),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(d_model, d_model),
            )
        
        # Stream 2: IMU feature extraction
        self.imu_feature_extractor = nn.Sequential(
            nn.Linear(imu_dim, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
        )
        
        # Positional encodings
        self.positional_encoder_pressure = PositionalEncoding(d_model)
        self.positional_encoder_imu = PositionalEncoding(d_model)
        
        # Two-stream transformer layers
        self.pressure_self_layers = nn.ModuleList()
        self.imu_self_layers = nn.ModuleList()
        self.pressure_to_imu_cross_layers = nn.ModuleList()
        self.imu_to_pressure_cross_layers = nn.ModuleList()
        self.pressure_norm_layers = nn.ModuleList()
        self.imu_norm_layers = nn.ModuleList()
        self.pressure_cross_norm_layers = nn.ModuleList()
        self.imu_cross_norm_layers = nn.ModuleList()
        
        for _ in range(num_encoder_layers):
            self.pressure_self_layers.append(
                nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
            )
            self.imu_self_layers.append(
                nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
            )
            self.pressure_to_imu_cross_layers.append(
                nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
            )
            self.imu_to_pressure_cross_layers.append(
                nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
            )
            self.pressure_norm_layers.append(nn.LayerNorm(d_model))
            self.imu_norm_layers.append(nn.LayerNorm(d_model))
            self.pressure_cross_norm_layers.append(nn.LayerNorm(d_model))
            self.imu_cross_norm_layers.append(nn.LayerNorm(d_model))
        
        # Fusion decoder
        self.fusion_decoder = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2),
            nn.LayerNorm(d_model * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, output_dim),
        )
    
    def forward(self, x):
        """Forward pass.
        
        Args:
            x: (batch, seq_len, pressure_dim + imu_dim)
        
        Returns:
            output: (batch, seq_len, output_dim)
        """
        # Split inputs
        pressure_x = x[..., :self.pressure_dim]
        imu_x = x[..., self.pressure_dim:self.pressure_dim + self.imu_dim]
        
        # Feature extraction
        pressure_feat = self.pressure_feature_extractor(pressure_x)
        imu_feat = self.imu_feature_extractor(imu_x)
        
        # Positional encoding
        pressure_feat = self.positional_encoder_pressure(pressure_feat)
        imu_feat = self.positional_encoder_imu(imu_feat)
        
        # Two-stream transformer
        for i in range(self.num_encoder_layers):
            # Self-attention
            pressure_self, _ = self.pressure_self_layers[i](pressure_feat, pressure_feat, pressure_feat)
            imu_self, _ = self.imu_self_layers[i](imu_feat, imu_feat, imu_feat)
            
            # Add & normalize
            pressure_feat = self.pressure_norm_layers[i](pressure_feat + pressure_self)
            imu_feat = self.imu_norm_layers[i](imu_feat + imu_self)
            
            # Cross-attention
            pressure_cross, _ = self.pressure_to_imu_cross_layers[i](pressure_feat, imu_feat, imu_feat)
            imu_cross, _ = self.imu_to_pressure_cross_layers[i](imu_feat, pressure_feat, pressure_feat)
            
            # Add & normalize
            pressure_feat = self.pressure_cross_norm_layers[i](pressure_feat + pressure_cross)
            imu_feat = self.imu_cross_norm_layers[i](imu_feat + imu_cross)
        
        # Fusion
        fused = torch.cat([pressure_feat, imu_feat], dim=-1)
        output = self.fusion_decoder(fused)
        
        return output

print("SoleFormer defined.")

SoleFormer defined.


In [ ]:
class AccelNet(nn.Module):
    """Auxiliary network for IMU cycle consistency."""
    
    def __init__(self, input_dim, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 12),  # 6DoF × 2 feet
        )
    
    def forward(self, x):
        """Predict IMU from pose.
        
        Args:
            x: (batch, seq_len, pose_dim) or (batch, pose_dim)
        
        Returns:
            imu: (batch, seq_len, 12) or (batch, 12)
        """
        return self.network(x)

class PressNet(nn.Module):
    """Auxiliary network for pressure cycle consistency."""
    
    def __init__(self, input_dim, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 32),  # 16 sensors × 2 feet
        )
    
    def forward(self, x):
        """Predict pressure from pose.
        
        Args:
            x: (batch, seq_len, pose_dim) or (batch, pose_dim)
        
        Returns:
            pressure: (batch, seq_len, 32) or (batch, 32)
        """
        return self.network(x)

print("AccelNet and PressNet defined.")

AccelNet and PressNet defined.


## 5. Loss Functions

In [5]:
# Utilities for EXPLICIT anthropometric constraints (direct subject values, no bone-length priors)

# Constraint bones are defined against the 23-joint layout used in this notebook.
# Order is aligned with anthropometric segment targets:
# 0: R upper arm, 1: R forearm, 2: L upper arm, 3: L forearm,
# 4: R thigh, 5: R shank, 6: L thigh, 7: L shank
CONSTRAINT_BONE_SPECS = [
    ((8, 9),   'R_Shoulder->R_Elbow'),
    ((9, 10),  'R_Elbow->R_Wrist'),
    ((12, 13), 'L_Shoulder->L_Elbow'),
    ((13, 14), 'L_Elbow->L_Wrist'),
    ((15, 16), 'R_Hip->R_Knee'),
    ((16, 17), 'R_Knee->R_Ankle'),
    ((19, 20), 'L_Hip->L_Knee'),
    ((20, 21), 'L_Knee->L_Ankle'),
]

SKELETON_KINETIC_CHAINS = [pair for pair, _ in CONSTRAINT_BONE_SPECS]

# Bone name order aligned with SKELETON_KINETIC_CHAINS
BONE_ORDER = [f'bone_{i}' for i in range(len(SKELETON_KINETIC_CHAINS))]
BONE_JOINT_LABELS = {f'bone_{i}': label for i, (_, label) in enumerate(CONSTRAINT_BONE_SPECS)}

# Bone-specific direct matching candidates.
BONE_FEATURE_CANDIDATES = {
    'bone_0': [['right', 'upper', 'arm'], ['right', 'humerus'], ['r', 'upper', 'arm']],
    'bone_1': [['right', 'forearm'], ['right', 'lower', 'arm'], ['right', 'radius'], ['right', 'ulna']],
    'bone_2': [['left', 'upper', 'arm'], ['left', 'humerus'], ['l', 'upper', 'arm']],
    'bone_3': [['left', 'forearm'], ['left', 'lower', 'arm'], ['left', 'radius'], ['left', 'ulna']],
    'bone_4': [['right', 'thigh'], ['right', 'femur'], ['right', 'upper', 'leg']],
    'bone_5': [['right', 'shank'], ['right', 'tibia'], ['right', 'lower', 'leg']],
    'bone_6': [['left', 'thigh'], ['left', 'femur'], ['left', 'upper', 'leg']],
    'bone_7': [['left', 'shank'], ['left', 'tibia'], ['left', 'lower', 'leg']],
}

GENERIC_SEGMENT_POOLS = {
    'upper_arm': [['upper', 'arm'], ['humerus']],
    'forearm': [['forearm'], ['lower', 'arm'], ['radius'], ['ulna']],
    'thigh': [['thigh'], ['femur'], ['upper', 'leg']],
    'shank': [['shank'], ['tibia'], ['lower', 'leg']],
}

def _tokenize_name(name):
    name = str(name).lower().replace('-', '_').replace(' ', '_')
    return [tok for tok in name.split('_') if tok]

def _find_feature_idx_by_tokens(feature_names, token_groups):
    """Find first feature index whose tokenized name contains all tokens of any group."""
    if feature_names is None:
        return None

    tokenized = [_tokenize_name(n) for n in feature_names]
    for tokens_required in token_groups:
        req = set(tokens_required)
        for i, toks in enumerate(tokenized):
            if req.issubset(set(toks)):
                return i
    return None

def _find_pool_indices(feature_names, token_groups):
    """Find all feature indices matching any token group."""
    if feature_names is None:
        return []
    tokenized = [_tokenize_name(n) for n in feature_names]
    out = []
    for i, toks in enumerate(tokenized):
        stoks = set(toks)
        for group in token_groups:
            if set(group).issubset(stoks):
                out.append(i)
                break
    return sorted(set(out))

def _get_named_feature_idx(feature_names, name_tokens):
    return _find_feature_idx_by_tokens(feature_names, [name_tokens])

def _derived_anthropo_rules(subject_anthropo, feature_names):
    """
    Build derived segment lengths from direct anthropometric columns when side-specific columns are absent.

    Returns:
        rules: bone -> {value, columns, formula} for available rules
    """
    rules = {}
    if subject_anthropo is None or feature_names is None:
        return rules

    idx_shoulder_width = _get_named_feature_idx(feature_names, ['shoulder', 'width'])
    idx_elbow_span = _get_named_feature_idx(feature_names, ['elbow', 'span'])
    idx_wrist_span = _get_named_feature_idx(feature_names, ['wrist', 'span'])
    idx_hip_height = _get_named_feature_idx(feature_names, ['hip', 'height'])
    idx_knee_height = _get_named_feature_idx(feature_names, ['knee', 'height'])
    idx_ankle_height = _get_named_feature_idx(feature_names, ['ankle', 'height'])

    # Upper-arm estimate: half of (elbow span - shoulder width)
    if idx_shoulder_width is not None and idx_elbow_span is not None:
        upper_arm = 0.5 * (subject_anthropo[:, idx_elbow_span] - subject_anthropo[:, idx_shoulder_width])
        upper_arm = torch.clamp(torch.abs(upper_arm), min=1e-6)
        rules['bone_0'] = {
            'value': upper_arm,
            'columns': [feature_names[idx_elbow_span], feature_names[idx_shoulder_width]],
            'formula': '0.5 * abs(elbow_span - shoulder_width)',
        }
        rules['bone_2'] = {
            'value': upper_arm,
            'columns': [feature_names[idx_elbow_span], feature_names[idx_shoulder_width]],
            'formula': '0.5 * abs(elbow_span - shoulder_width)',
        }

    # Forearm estimate: half of (wrist span - elbow span)
    if idx_wrist_span is not None and idx_elbow_span is not None:
        forearm = 0.5 * (subject_anthropo[:, idx_wrist_span] - subject_anthropo[:, idx_elbow_span])
        forearm = torch.clamp(torch.abs(forearm), min=1e-6)
        rules['bone_1'] = {
            'value': forearm,
            'columns': [feature_names[idx_wrist_span], feature_names[idx_elbow_span]],
            'formula': '0.5 * abs(wrist_span - elbow_span)',
        }
        rules['bone_3'] = {
            'value': forearm,
            'columns': [feature_names[idx_wrist_span], feature_names[idx_elbow_span]],
            'formula': '0.5 * abs(wrist_span - elbow_span)',
        }

    # Thigh estimate: hip_height - knee_height
    if idx_hip_height is not None and idx_knee_height is not None:
        thigh = subject_anthropo[:, idx_hip_height] - subject_anthropo[:, idx_knee_height]
        thigh = torch.clamp(torch.abs(thigh), min=1e-6)
        rules['bone_4'] = {
            'value': thigh,
            'columns': [feature_names[idx_hip_height], feature_names[idx_knee_height]],
            'formula': 'abs(hip_height - knee_height)',
        }
        rules['bone_6'] = {
            'value': thigh,
            'columns': [feature_names[idx_hip_height], feature_names[idx_knee_height]],
            'formula': 'abs(hip_height - knee_height)',
        }

    # Shank estimate: knee_height - ankle_height
    if idx_knee_height is not None and idx_ankle_height is not None:
        shank = subject_anthropo[:, idx_knee_height] - subject_anthropo[:, idx_ankle_height]
        shank = torch.clamp(torch.abs(shank), min=1e-6)
        rules['bone_5'] = {
            'value': shank,
            'columns': [feature_names[idx_knee_height], feature_names[idx_ankle_height]],
            'formula': 'abs(knee_height - ankle_height)',
        }
        rules['bone_7'] = {
            'value': shank,
            'columns': [feature_names[idx_knee_height], feature_names[idx_ankle_height]],
            'formula': 'abs(knee_height - ankle_height)',
        }

    return rules

def describe_bone_feature_mapping(feature_names):
    """Print a transparent mapping report: each constrained bone -> anthropometric source columns."""
    mapping = {}
    if feature_names is None:
        print('Bone mapping report: no anthropometric feature names available.')
        return mapping

    fallback_map = {
        'bone_0': 'upper_arm', 'bone_2': 'upper_arm',
        'bone_1': 'forearm',  'bone_3': 'forearm',
        'bone_4': 'thigh',    'bone_6': 'thigh',
        'bone_5': 'shank',    'bone_7': 'shank',
    }

    pool_cache = {
        seg: _find_pool_indices(feature_names, groups)
        for seg, groups in GENERIC_SEGMENT_POOLS.items()
    }

    derived_stub = _derived_anthropo_rules(
        subject_anthropo=torch.ones((1, len(feature_names))),
        feature_names=feature_names,
    )

    for bone_name in BONE_ORDER:
        direct_idx = _find_feature_idx_by_tokens(feature_names, BONE_FEATURE_CANDIDATES.get(bone_name, []))
        if direct_idx is not None:
            mapping[bone_name] = {
                'mode': 'direct',
                'indices': [direct_idx],
                'columns': [feature_names[direct_idx]],
                'formula': None,
            }
            continue

        seg = fallback_map.get(bone_name, None)
        idxs = pool_cache.get(seg, []) if seg is not None else []
        if idxs:
            mapping[bone_name] = {
                'mode': f'pool:{seg}',
                'indices': idxs,
                'columns': [feature_names[i] for i in idxs],
                'formula': 'mean(pool_columns)',
            }
            continue

        if bone_name in derived_stub:
            mapping[bone_name] = {
                'mode': 'derived',
                'indices': [],
                'columns': derived_stub[bone_name]['columns'],
                'formula': derived_stub[bone_name]['formula'],
            }
        else:
            mapping[bone_name] = {
                'mode': 'missing',
                'indices': [],
                'columns': [],
                'formula': None,
            }

    print('\nBone mapping report (direct subject anthropometrics + constrained joints):')
    for b in BONE_ORDER:
        m = mapping[b]
        joint_label = BONE_JOINT_LABELS.get(b, 'unknown-joints')
        if m['mode'] == 'missing':
            print(f"  {b} [{joint_label}]: MISSING (no matching anthropometric column)")
        elif m['mode'] == 'derived':
            print(f"  {b} [{joint_label}]: derived -> columns={m['columns']} | formula={m['formula']}")
        else:
            print(f"  {b} [{joint_label}]: {m['mode']} -> {m['columns']}")
    return mapping

def anthropo_to_target_bone_lengths(subject_anthropo, feature_names=None):
    """
    Convert direct subject anthropometrics into per-bone targets.

    No hardcoded numeric length priors are used.
    Values are taken directly from anthropometric channels when available.

    Args:
        subject_anthropo: Tensor (batch, n_features), in direct/original units.
        feature_names: list[str]

    Returns:
        Dict[str, Tensor(batch,)] with None for missing bones.
    """
    targets = {b: None for b in BONE_ORDER}

    if subject_anthropo is None or subject_anthropo.numel() == 0:
        return targets
    if subject_anthropo.dim() != 2:
        raise ValueError(f"subject_anthropo must be (batch, n_features), got {tuple(subject_anthropo.shape)}")

    # Direct side-specific extraction where possible
    for bone_name in BONE_ORDER:
        idx = _find_feature_idx_by_tokens(feature_names, BONE_FEATURE_CANDIDATES.get(bone_name, []))
        if idx is not None and idx < subject_anthropo.shape[1]:
            targets[bone_name] = subject_anthropo[:, idx]

    # Fallback to segment pools (still direct measurements, aggregated from available channels)
    pool_cache = {
        seg: _find_pool_indices(feature_names, groups)
        for seg, groups in GENERIC_SEGMENT_POOLS.items()
    }

    fallback_map = {
        'bone_0': 'upper_arm', 'bone_2': 'upper_arm',
        'bone_1': 'forearm',  'bone_3': 'forearm',
        'bone_4': 'thigh',    'bone_6': 'thigh',
        'bone_5': 'shank',    'bone_7': 'shank',
    }

    for bone_name, seg in fallback_map.items():
        if targets[bone_name] is not None:
            continue
        idxs = [i for i in pool_cache.get(seg, []) if i < subject_anthropo.shape[1]]
        if idxs:
            targets[bone_name] = subject_anthropo[:, idxs].mean(dim=1)

    # Derived direct-anthropometric rules (column arithmetic, no priors)
    derived_rules = _derived_anthropo_rules(subject_anthropo, feature_names)
    for bone_name, rule in derived_rules.items():
        if targets[bone_name] is None:
            targets[bone_name] = rule['value']

    return targets

def compute_bone_lengths(pose):
    """
    Compute per-sample predicted bone lengths.

    Args:
        pose: (batch, seq_len, D) or (batch, D)

    Returns:
        Dict[str, Tensor(batch,)]
    """
    if pose.dim() == 3:
        batch_size, seq_len, d = pose.shape
        if d % 3 != 0:
            raise ValueError(f"Pose feature dimension must be divisible by 3, got {d}")
        n_joints = d // 3
        joints = pose.reshape(batch_size, seq_len, n_joints, 3)

        out = {}
        for i, (j1, j2) in enumerate(SKELETON_KINETIC_CHAINS):
            if j1 >= n_joints or j2 >= n_joints:
                continue
            diff = joints[:, :, j1, :] - joints[:, :, j2, :]
            dists = torch.linalg.norm(diff, dim=-1)
            out[f'bone_{i}'] = dists.mean(dim=1)
        return out

    if pose.dim() == 2:
        batch_size, d = pose.shape
        if d % 3 != 0:
            raise ValueError(f"Pose feature dimension must be divisible by 3, got {d}")
        n_joints = d // 3
        joints = pose.reshape(batch_size, n_joints, 3)

        out = {}
        for i, (j1, j2) in enumerate(SKELETON_KINETIC_CHAINS):
            if j1 >= n_joints or j2 >= n_joints:
                continue
            diff = joints[:, j1, :] - joints[:, j2, :]
            out[f'bone_{i}'] = torch.linalg.norm(diff, dim=-1)
        return out

    raise ValueError(f"pose must be rank 2 or 3, got rank {pose.dim()}")

def apply_bone_length_constraints(pred_pose, target_bone_lengths, weight=0.1):
    """
    Compute bone-length consistency loss.

    Args:
        pred_pose: (batch, seq_len, D)
        target_bone_lengths: Dict[str, Tensor(batch,)]
        weight: scalar

    Returns:
        Scalar tensor
    """
    if not isinstance(target_bone_lengths, dict):
        return torch.tensor(0.0, device=pred_pose.device)

    pred_lengths = compute_bone_lengths(pred_pose)
    if not pred_lengths:
        return torch.tensor(0.0, device=pred_pose.device)

    total = torch.tensor(0.0, device=pred_pose.device)
    valid_terms = 0

    for bone_name, pred_len in pred_lengths.items():
        target_len = target_bone_lengths.get(bone_name, None)
        if target_len is None:
            continue

        if not torch.is_tensor(target_len):
            target_len = torch.tensor(target_len, dtype=pred_len.dtype, device=pred_len.device)
        else:
            target_len = target_len.to(device=pred_len.device, dtype=pred_len.dtype)

        target_len = target_len.reshape(-1)
        pred_len = pred_len.reshape(-1)
        if target_len.shape[0] != pred_len.shape[0]:
            min_n = min(target_len.shape[0], pred_len.shape[0])
            target_len = target_len[:min_n]
            pred_len = pred_len[:min_n]

        if target_len.numel() == 0:
            continue

        total = total + F.mse_loss(pred_len, target_len)
        valid_terms += 1

    if valid_terms == 0:
        return torch.tensor(0.0, device=pred_pose.device)

    return weight * (total / valid_terms)

print("Explicit anthropometric constraints now use direct subject anthropometric values (no numeric bone priors).")
if 'anthropo_feature_names' in globals() and anthropo_feature_names:
    _BONE_MAPPING_REPORT = describe_bone_feature_mapping(anthropo_feature_names)

Explicit anthropometric constraints now use direct subject anthropometric values (no numeric bone priors).


In [ ]:
def rescale_skeleton_to_anthropometry(pred_pose_flat, subject_anthropo, feature_names=None, pose_scaler=None):
    """Rescale predicted skeletons to anthropometric target segment lengths.

    Args:
        pred_pose_flat: ndarray (n_frames, pose_dim) in standardized space if pose_scaler is provided,
            otherwise assumed to already be in physical/original units.
        subject_anthropo: ndarray or Tensor (n_rows, n_anthro) in original anthropometric units.
        feature_names: anthropometric column names.
        pose_scaler: optional fitted scaler used for pose normalization.

    Returns:
        ndarray (n_frames, pose_dim), same space as input pred_pose_flat.
    """
    pred_arr = np.asarray(pred_pose_flat, dtype=np.float32)
    if pred_arr.ndim != 2:
        raise ValueError(f"pred_pose_flat must be 2D (n_frames, pose_dim), got shape {pred_arr.shape}")

    if subject_anthropo is None:
        return pred_arr

    anth = subject_anthropo.detach().cpu().numpy() if torch.is_tensor(subject_anthropo) else np.asarray(subject_anthropo)
    if anth.ndim != 2 or anth.shape[0] == 0:
        return pred_arr

    n_frames, pose_dim = pred_arr.shape
    if pose_dim % 3 != 0:
        raise ValueError(f"pose_dim must be divisible by 3, got {pose_dim}")
    n_joints = pose_dim // 3

    # Align anthropometric rows to prediction rows when counts differ.
    if anth.shape[0] != n_frames:
        idx = np.linspace(0, anth.shape[0] - 1, num=n_frames)
        idx = np.clip(np.round(idx).astype(int), 0, anth.shape[0] - 1)
        anth = anth[idx]

    # Work in physical/original units when scaler is available.
    if pose_scaler is not None:
        pose_phys = pose_scaler.inverse_transform(pred_arr)
    else:
        pose_phys = pred_arr.copy()

    pose_t = torch.from_numpy(pose_phys.astype(np.float32)).reshape(n_frames, n_joints, 3)
    anth_t = torch.from_numpy(anth.astype(np.float32))

    target_bones = anthropo_to_target_bone_lengths(anth_t, feature_names=feature_names)

    out = pose_t.clone()
    eps = 1e-6
    for i, (j1, j2) in enumerate(SKELETON_KINETIC_CHAINS):
        if j1 >= n_joints or j2 >= n_joints:
            continue
        tgt = target_bones.get(f'bone_{i}', None)
        if tgt is None:
            continue
        tgt = tgt.reshape(-1).to(dtype=out.dtype)
        if tgt.shape[0] != n_frames:
            m = min(tgt.shape[0], n_frames)
            tgt = tgt[:m]
            if m < n_frames:
                pad = tgt[-1].repeat(n_frames - m)
                tgt = torch.cat([tgt, pad], dim=0)

        v = out[:, j2, :] - out[:, j1, :]
        cur = torch.linalg.norm(v, dim=-1).clamp_min(eps)
        scale = torch.clamp(tgt / cur, min=0.25, max=4.0).unsqueeze(-1)
        out[:, j2, :] = out[:, j1, :] + v * scale

    out_flat = out.reshape(n_frames, pose_dim).cpu().numpy()
    if pose_scaler is not None:
        out_flat = pose_scaler.transform(out_flat)

    return out_flat.astype(np.float32)

print('Defined rescale_skeleton_to_anthropometry().')

Defined rescale_skeleton_to_anthropometry().


In [7]:
# Unit-consistency override for explicit anthropometric constraints/rescaling
# Confirmed dataset convention: anthropometric lengths are stored in centimeters.
# Pose coordinates are treated in meters after de-normalization.

ANTHROPO_UNIT = 'cm'
POSE_UNIT = 'm'
CM_TO_M = 0.01


def _estimate_anthropo_unit_scale(subject_anthropo=None, feature_names=None):
    """Return multiplier to convert anthropometric lengths into pose units."""
    if ANTHROPO_UNIT == 'cm' and POSE_UNIT == 'm':
        return CM_TO_M
    return 1.0


def _scaled_subject_anthropo(subject_anthropo, feature_names=None):
    scale = _estimate_anthropo_unit_scale(subject_anthropo, feature_names=feature_names)
    if torch.is_tensor(subject_anthropo):
        return subject_anthropo * scale, scale
    return torch.as_tensor(subject_anthropo, dtype=torch.float32) * scale, scale


# Override with unit-aware mapping
def anthropo_to_target_bone_lengths(subject_anthropo, feature_names=None):
    targets = {b: None for b in BONE_ORDER}

    if subject_anthropo is None:
        return targets

    subject_anthropo, _ = _scaled_subject_anthropo(subject_anthropo, feature_names=feature_names)

    if subject_anthropo.numel() == 0:
        return targets
    if subject_anthropo.dim() != 2:
        raise ValueError(f"subject_anthropo must be (batch, n_features), got {tuple(subject_anthropo.shape)}")

    for bone_name in BONE_ORDER:
        idx = _find_feature_idx_by_tokens(feature_names, BONE_FEATURE_CANDIDATES.get(bone_name, []))
        if idx is not None and idx < subject_anthropo.shape[1]:
            targets[bone_name] = subject_anthropo[:, idx]

    pool_cache = {
        seg: _find_pool_indices(feature_names, groups)
        for seg, groups in GENERIC_SEGMENT_POOLS.items()
    }

    fallback_map = {
        'bone_0': 'upper_arm', 'bone_2': 'upper_arm',
        'bone_1': 'forearm',  'bone_3': 'forearm',
        'bone_4': 'thigh',    'bone_6': 'thigh',
        'bone_5': 'shank',    'bone_7': 'shank',
    }

    for bone_name, seg in fallback_map.items():
        if targets[bone_name] is not None:
            continue
        idxs = [i for i in pool_cache.get(seg, []) if i < subject_anthropo.shape[1]]
        if idxs:
            targets[bone_name] = subject_anthropo[:, idxs].mean(dim=1)

    derived_rules = _derived_anthropo_rules(subject_anthropo, feature_names)
    for bone_name, rule in derived_rules.items():
        if targets[bone_name] is None:
            targets[bone_name] = rule['value']

    return targets


# Override with unit-aware post-processing rescaler
def rescale_skeleton_to_anthropometry(pred_pose_flat, subject_anthropo, feature_names=None, pose_scaler=None):
    pred_arr = np.asarray(pred_pose_flat, dtype=np.float32)
    if pred_arr.ndim != 2:
        raise ValueError(f"pred_pose_flat must be 2D (n_frames, pose_dim), got shape {pred_arr.shape}")

    if subject_anthropo is None:
        return pred_arr

    anth = subject_anthropo.detach().cpu().numpy() if torch.is_tensor(subject_anthropo) else np.asarray(subject_anthropo)
    if anth.ndim != 2 or anth.shape[0] == 0:
        return pred_arr

    n_frames, pose_dim = pred_arr.shape
    if pose_dim % 3 != 0:
        raise ValueError(f"pose_dim must be divisible by 3, got {pose_dim}")
    n_joints = pose_dim // 3

    if anth.shape[0] != n_frames:
        idx = np.linspace(0, anth.shape[0] - 1, num=n_frames)
        idx = np.clip(np.round(idx).astype(int), 0, anth.shape[0] - 1)
        anth = anth[idx]

    if pose_scaler is not None:
        pose_phys = pose_scaler.inverse_transform(pred_arr)
    else:
        pose_phys = pred_arr.copy()

    pose_t = torch.from_numpy(pose_phys.astype(np.float32)).reshape(n_frames, n_joints, 3)
    anth_t = torch.from_numpy(anth.astype(np.float32))

    unit_scale = _estimate_anthropo_unit_scale(anth_t, feature_names=feature_names)
    target_bones = anthropo_to_target_bone_lengths(anth_t, feature_names=feature_names)

    out = pose_t.clone()
    eps = 1e-6
    for i, (j1, j2) in enumerate(SKELETON_KINETIC_CHAINS):
        if j1 >= n_joints or j2 >= n_joints:
            continue
        tgt = target_bones.get(f'bone_{i}', None)
        if tgt is None:
            continue
        tgt = tgt.reshape(-1).to(dtype=out.dtype)
        if tgt.shape[0] != n_frames:
            m = min(tgt.shape[0], n_frames)
            tgt = tgt[:m]
            if m < n_frames:
                pad = tgt[-1].repeat(n_frames - m)
                tgt = torch.cat([tgt, pad], dim=0)

        v = out[:, j2, :] - out[:, j1, :]
        cur = torch.linalg.norm(v, dim=-1).clamp_min(eps)
        scale = torch.clamp(tgt / cur, min=0.25, max=4.0).unsqueeze(-1)
        out[:, j2, :] = out[:, j1, :] + v * scale

    out_flat = out.reshape(n_frames, pose_dim).cpu().numpy()
    if pose_scaler is not None:
        out_flat = pose_scaler.transform(out_flat)

    print(f"Anthropometric unit scale applied: {unit_scale} ({ANTHROPO_UNIT}->{POSE_UNIT})")
    return out_flat.astype(np.float32)


print('Applied fixed cm->m unit-consistency overrides for anthropometric constraints and rescaling.')

Applied fixed cm->m unit-consistency overrides for anthropometric constraints and rescaling.


In [32]:
class DoubleCycleConsistencyLoss(nn.Module):
    """Double-cycle consistency loss with EXPLICIT anthropometric constraints.

    Loss = w_pose * L_pose + w_imu * L_imu_cycle + w_pressure * L_pressure_cycle + w_bone * L_bone_constraint

    EXPLICIT LEARNING: bone targets are derived from subject anthropometric vectors.
    """

    def __init__(
        self,
        accel_net=None,
        press_net=None,
        weight_pose=1.0,
        weight_imu_cycle=0.5,
        weight_pressure_cycle=0.5,
        weight_bone_constraint=0.1,
        anthropo_feature_names=None,
    ):
        super().__init__()
        self.accel_net = accel_net
        self.press_net = press_net
        self.weight_pose = weight_pose
        self.weight_imu_cycle = weight_imu_cycle
        self.weight_pressure_cycle = weight_pressure_cycle
        self.weight_bone_constraint = weight_bone_constraint
        self.anthropo_feature_names = anthropo_feature_names
        self.mse_loss = nn.MSELoss()

    def forward(self, pred_pose, target_pose, input_imu=None, input_pressure=None, subject_anthropo=None):
        """Compute double-cycle consistency loss with explicit bone constraints.

        Args:
            pred_pose: (batch, seq_len, D) - Predicted pose
            target_pose: (batch, seq_len, D) - Ground truth pose
            input_imu: (batch, seq_len, 12) - Input IMU (optional)
            input_pressure: (batch, seq_len, 32) - Input pressure (optional)
            subject_anthropo: (batch, n_anthro) - Subject anthropometric vectors

        Returns:
            total_loss: scalar
            loss_dict: dictionary of components
        """
        # 1. Pose loss
        pose_loss = self.mse_loss(pred_pose, target_pose)
        total_loss = self.weight_pose * pose_loss
        loss_dict = {'pose_loss': pose_loss.item()}

        # 2. IMU cycle loss
        if self.accel_net is not None and input_imu is not None:
            pred_imu = self.accel_net(pred_pose)
            if pred_imu.dim() == 3:
                pred_imu = pred_imu.reshape(pred_imu.shape[0], -1)
            if input_imu.dim() == 3:
                input_imu_flat = input_imu.reshape(input_imu.shape[0], -1)
            else:
                input_imu_flat = input_imu
            imu_cycle_loss = self.mse_loss(pred_imu, input_imu_flat)
            total_loss = total_loss + self.weight_imu_cycle * imu_cycle_loss
            loss_dict['imu_cycle_loss'] = imu_cycle_loss.item()

        # 3. Pressure cycle loss
        if self.press_net is not None and input_pressure is not None:
            pred_pressure = self.press_net(pred_pose)
            if pred_pressure.dim() == 3:
                pred_pressure = pred_pressure.reshape(pred_pressure.shape[0], -1)
            if input_pressure.dim() == 3:
                input_pressure_flat = input_pressure.reshape(input_pressure.shape[0], -1)
            else:
                input_pressure_flat = input_pressure
            pressure_cycle_loss = self.mse_loss(pred_pressure, input_pressure_flat)
            total_loss = total_loss + self.weight_pressure_cycle * pressure_cycle_loss
            loss_dict['pressure_cycle_loss'] = pressure_cycle_loss.item()

        # 4. Bone-length constraint loss using mapped anthropometric targets
        if subject_anthropo is not None and subject_anthropo.numel() > 0:
            target_bone_lengths = anthropo_to_target_bone_lengths(
                subject_anthropo,
                feature_names=self.anthropo_feature_names,
            )
            bone_constraint_loss = apply_bone_length_constraints(
                pred_pose,
                target_bone_lengths,
                weight=1.0,
            )
            total_loss = total_loss + self.weight_bone_constraint * bone_constraint_loss
            loss_dict['bone_constraint_loss'] = bone_constraint_loss.item()

        loss_dict['total_loss'] = total_loss.item()
        return total_loss, loss_dict

print("DoubleCycleConsistencyLoss now uses anthropometric-to-bone target mapping.")

DoubleCycleConsistencyLoss now uses anthropometric-to-bone target mapping.


## 6. Training Pipeline

In [ ]:
# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Infer output structure from loaded TRAIN pose data
if pose_train_processed.shape[1] % 3 != 0:
    raise ValueError(
        f"Pose feature dimension must be divisible by 3, got {pose_train_processed.shape[1]}"
    )
num_joints = pose_train_processed.shape[1] // 3
# EXPLICIT LEARNING: Do NOT include anthropometric dimensions in input
anthropo_dim = 0  # Anthropometrics are NOT input to model
effective_imu_dim = imu_train_processed.shape[1]  # Just IMU: 12 dims
print(f"Detected target joints: {num_joints}, output_dim: {pose_train_processed.shape[1]}")
print(f"Input IMU dimension: {effective_imu_dim} (anthropometrics NOT in input)")
print(f"Model will use EXPLICIT anthropometric constraints via loss function and post-processing")

# Model configuration (based on TRAIN split feature dims)
# Input: 32 pressure + 12 IMU = 44 dims (no anthropometrics)
model_config = {
    'pressure_dim': pressure_train_processed.shape[1],
    'imu_dim': effective_imu_dim,
    'd_model': 128,
    'nhead': 8,
    'num_encoder_layers': 2,
    'output_dim': pose_train_processed.shape[1],
    'dropout': 0.1,
}

# Training configuration
train_config = {
    'num_epochs': 50,
    'batch_size': 32,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'warmup_epochs': 5,
}

print("\nModel Configuration:")
for k, v in model_config.items():
    print(f"  {k}: {v}")

print("\nTraining Configuration:")
for k, v in train_config.items():
    print(f"  {k}: {v}")

Device: cpu
Detected target joints: 23, output_dim: 69
Anthropometric conditioning dim: 11

Model Configuration:
  pressure_dim: 32
  imu_dim: 23
  d_model: 128
  nhead: 8
  num_encoder_layers: 2
  output_dim: 69
  dropout: 0.1

Training Configuration:
  num_epochs: 50
  batch_size: 32
  learning_rate: 0.001
  weight_decay: 1e-05
  warmup_epochs: 5


In [ ]:
# Initialize models
model = SoleFormer(**model_config).to(device)
accel_net = AccelNet(input_dim=model_config['output_dim'], hidden_dim=256, dropout=0.1).to(device)
press_net = PressNet(input_dim=model_config['output_dim'], hidden_dim=256, dropout=0.1).to(device)

# Pre-train AccelNet and PressNet (optional - can do actual pre-training)
print("Initializing auxiliary networks...")
print(f"SoleFormer parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"AccelNet parameters: {sum(p.numel() for p in accel_net.parameters()):,}")
print(f"PressNet parameters: {sum(p.numel() for p in press_net.parameters()):,}")

Initializing auxiliary networks...
SoleFormer parameters: 740,421
AccelNet parameters: 87,820
PressNet parameters: 92,960


In [33]:
# Loss function and optimizer with EXPLICIT anthropometric constraints
criterion = DoubleCycleConsistencyLoss(
    accel_net=accel_net,
    press_net=press_net,
    weight_pose=1.0,
    weight_imu_cycle=0.5,
    weight_pressure_cycle=0.5,
    weight_bone_constraint=0.1,
    anthropo_feature_names=anthropo_feature_names if 'anthropo_feature_names' in globals() else None,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_config['learning_rate'],
    weight_decay=train_config['weight_decay'],
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=train_config['num_epochs'],
)

print("Optimizer and scheduler initialized (explicit anthropometric bone targets enabled).")

Optimizer and scheduler initialized (explicit anthropometric bone targets enabled).


In [31]:
# Training function with explicit anthropometric constraints
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    loss_components = {'pose_loss': 0.0, 'imu_cycle_loss': 0.0, 'pressure_cycle_loss': 0.0, 'bone_constraint_loss': 0.0}

    for batch in tqdm(train_loader, desc='Training'):
        input_seq = batch['input'].to(device)
        pressure_seq = batch['pressure'].to(device)
        imu_seq = batch['imu'].to(device)
        pose_seq = batch['pose'].to(device)

        # Direct per-sample anthropometrics from DataLoader (subject-key aligned)
        subject_anthropo_batch = None
        if 'anthropo' in batch and batch['anthropo'].numel() > 0:
            subject_anthropo_batch = batch['anthropo'].to(device)

        optimizer.zero_grad()

        # Forward pass
        pred_pose = model(input_seq)

        # Compute loss with explicit constraints
        loss, loss_dict = criterion(
            pred_pose,
            pose_seq,
            imu_seq,
            pressure_seq,
            subject_anthropo=subject_anthropo_batch,
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        for key in loss_components:
            if key in loss_dict:
                loss_components[key] += loss_dict[key]

    avg_loss = total_loss / len(train_loader)
    for key in loss_components:
        loss_components[key] /= len(train_loader)

    return avg_loss, loss_components

def val_epoch(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    loss_components = {'pose_loss': 0.0, 'imu_cycle_loss': 0.0, 'pressure_cycle_loss': 0.0, 'bone_constraint_loss': 0.0}

    with torch.no_grad():
        for batch in tqdm(val_loader, desc='Validation'):
            input_seq = batch['input'].to(device)
            pressure_seq = batch['pressure'].to(device)
            imu_seq = batch['imu'].to(device)
            pose_seq = batch['pose'].to(device)

            subject_anthropo_batch = None
            if 'anthropo' in batch and batch['anthropo'].numel() > 0:
                subject_anthropo_batch = batch['anthropo'].to(device)

            pred_pose = model(input_seq)
            loss, loss_dict = criterion(
                pred_pose,
                pose_seq,
                imu_seq,
                pressure_seq,
                subject_anthropo=subject_anthropo_batch,
            )

            total_loss += loss.item()
            for key in loss_components:
                if key in loss_dict:
                    loss_components[key] += loss_dict[key]

    avg_loss = total_loss / len(val_loader)
    for key in loss_components:
        loss_components[key] /= len(val_loader)

    return avg_loss, loss_components

print("Training functions now use batch-aligned direct anthropometrics from dataset samples.")

Training functions now use batch-aligned direct anthropometrics from dataset samples.


In [ ]:
from pathlib import Path

# Always write results under project root (not notebook working directory)
SOLEFORMER_RESULTS_DIR = PROJECT_ROOT / 'results' / 'SoleFormer'
MODEL_RESULTS_DIR = SOLEFORMER_RESULTS_DIR / 'model'
TRAINING_RESULTS_DIR = SOLEFORMER_RESULTS_DIR / 'training_results'
PREDICTION_RESULTS_DIR = SOLEFORMER_RESULTS_DIR / 'prediction'
ANIMATION_RESULTS_DIR = SOLEFORMER_RESULTS_DIR / 'animation'

# Create output directories
output_dirs = [
    MODEL_RESULTS_DIR,
    TRAINING_RESULTS_DIR,
    PREDICTION_RESULTS_DIR,
    ANIMATION_RESULTS_DIR,
]
for d in output_dirs:
    d.mkdir(parents=True, exist_ok=True)

if anthropo_train_raw is not None and len(anthropo_train_raw) > 0:
    print(f"Using EXPLICIT anthropometric constraints with direct subject values, shape {anthropo_train_raw.shape}")
else:
    print("No anthropometric data available. Bone-length constraints will be skipped.")

# Training loop
print(f"\nStarting training for {train_config['num_epochs']} epochs...")
print("Learning method: EXPLICIT (direct subject anthropometrics for bone targets)\n")

best_val_loss = float('inf')
train_losses = []
val_losses = []
train_loss_components = {key: [] for key in ['pose_loss', 'imu_cycle_loss', 'pressure_cycle_loss', 'bone_constraint_loss']}
val_loss_components = {key: [] for key in ['pose_loss', 'imu_cycle_loss', 'pressure_cycle_loss', 'bone_constraint_loss']}

model_save_path = MODEL_RESULTS_DIR / 'soleformer_best.pt'

for epoch in range(train_config['num_epochs']):
    train_loss, train_comp = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_comp = val_epoch(model, val_loader, criterion, device)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    for key in train_loss_components:
        if key in train_comp:
            train_loss_components[key].append(train_comp[key])
    for key in val_loss_components:
        if key in val_comp:
            val_loss_components[key].append(val_comp[key])

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), str(model_save_path))

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{train_config['num_epochs']}")
        print(f"  Train Loss: {train_loss:.6f}")
        print(f"  Val Loss: {val_loss:.6f}")
        print(f"  Components - Pose: {val_comp.get('pose_loss', 0):.6f}, IMU: {val_comp.get('imu_cycle_loss', 0):.6f}, Pressure: {val_comp.get('pressure_cycle_loss', 0):.6f}, Bone: {val_comp.get('bone_constraint_loss', 0):.6f}")

print(f"\nTraining completed! Best validation loss: {best_val_loss:.6f}")

## 7. Results Visualization

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Total loss
axes[0, 0].plot(train_losses, label='Train', marker='o', markersize=3)
axes[0, 0].plot(val_losses, label='Val', marker='s', markersize=3)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Total Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Pose loss
if train_loss_components['pose_loss']:
    axes[0, 1].plot(train_loss_components['pose_loss'], label='Train', marker='o', markersize=3)
    axes[0, 1].plot(val_loss_components['pose_loss'], label='Val', marker='s', markersize=3)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].set_title('Pose Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

# IMU cycle loss
if train_loss_components['imu_cycle_loss']:
    axes[1, 0].plot(train_loss_components['imu_cycle_loss'], label='Train', marker='o', markersize=3)
    axes[1, 0].plot(val_loss_components['imu_cycle_loss'], label='Val', marker='s', markersize=3)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].set_title('IMU Cycle Loss')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

# Pressure cycle loss
if train_loss_components['pressure_cycle_loss']:
    axes[1, 1].plot(train_loss_components['pressure_cycle_loss'], label='Train', marker='o', markersize=3)
    axes[1, 1].plot(val_loss_components['pressure_cycle_loss'], label='Val', marker='s', markersize=3)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Loss')
    axes[1, 1].set_title('Pressure Cycle Loss')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
training_fig_path = TRAINING_RESULTS_DIR / 'training_curves.png'
plt.savefig(str(training_fig_path), dpi=150, bbox_inches='tight')
plt.show()

print(f"Training curves saved to '{training_fig_path}'")

## 8. Inference and Evaluation

In [3]:
# Define joint names
JOINT_NAMES = [
    "Pelvis",
    "L5-S1",
    "L4-L3",
    "L1-T12",
    "T9-T8",
    "T1-C7",
    "C1-Head",
    "R_C7-Shoulder",
    "R_Shoulder",
    "R_Elbow",
    "R_Wrist",
    "L_C7-Shoulder",
    "L_Shoulder",
    "L_Elbow",
    "L_Wrist",
    "R_Hip",
    "R_Knee",
    "R_Ankle",
    "R_Ballfoot",
    "L_Hip",
    "L_Knee",
    "L_Ankle",
    "L_Ballfoot"
]

def resolve_test_result_tag(project_root):
    """Build a file-safe tag from top-level test skeleton filenames."""
    import re
    import unicodedata

    test_sk_dir = Path(project_root) / 'data' / 'test_data' / 'skeleton'
    tags = []
    if test_sk_dir.exists():
        for p in sorted(test_sk_dir.glob('Awinda_*.csv')):
            tag = p.stem[len('Awinda_'):]
            tag = unicodedata.normalize('NFKD', tag).encode('ascii', 'ignore').decode('ascii')
            tag = re.sub(r'[^A-Za-z0-9_-]+', '_', tag).strip('_')
            if tag:
                tags.append(tag)

    if not tags:
        return 'test_unknown'
    if len(tags) == 1:
        return tags[0]
    return f"{tags[0]}_plus{len(tags)-1}"

TEST_RESULT_TAG = resolve_test_result_tag(PROJECT_ROOT)
print(f"Test result tag: {TEST_RESULT_TAG}")

# Load best model
model.load_state_dict(torch.load(str(model_save_path)))
model.eval()

# Enhanced evaluation metrics
def compute_detailed_metrics(pred, target, scaler=None):
    """
    Compute comprehensive metrics:
    - Per-joint MPJPE with mean and std dev
    - Global MPJPE with mean and std dev
    - Procrustes Aligned-MPJPE (reconstruction error)
    - MPJ velocity error (smoothness)
    Returns: dict with all metrics
    """
    if isinstance(pred, torch.Tensor):
        pred = pred.detach().cpu().numpy()
    if isinstance(target, torch.Tensor):
        target = target.detach().cpu().numpy()

    b, t, d = pred.shape
    if d % 3 != 0:
        raise ValueError(f"Pose dimension must be divisible by 3, got {d}")
    joint_count = d // 3

    pred_flat = pred.reshape(-1, d)
    target_flat = target.reshape(-1, d)

    if scaler is not None:
        pred_denorm = scaler.inverse_transform(pred_flat)
        target_denorm = scaler.inverse_transform(target_flat)
    else:
        pred_denorm = pred_flat
        target_denorm = target_flat

    pred_joints = pred_denorm.reshape(-1, joint_count, 3)
    target_joints = target_denorm.reshape(-1, joint_count, 3)

    # Per-joint MPJPE
    per_joint_error = np.linalg.norm(pred_joints - target_joints, axis=-1)  # (n_frames, n_joints)
    per_joint_mpjpe = per_joint_error.mean(axis=0)  # Mean across frames
    per_joint_std = per_joint_error.std(axis=0)    # Std dev across frames

    # Global MPJPE
    global_mpjpe = per_joint_mpjpe.mean()
    global_std = per_joint_error.mean(axis=1).std()  # Std of frame-wise means

    # Procrustes Aligned-MPJPE (Reconstruction Error)
    procrustes_errors = []
    for frame_pred, frame_target in zip(pred_joints, target_joints):
        frame_pred_c = frame_pred - frame_pred.mean(axis=0)
        frame_target_c = frame_target - frame_target.mean(axis=0)

        H = frame_pred_c.T @ frame_target_c
        U, S, Vt = np.linalg.svd(H)
        R = Vt.T @ U.T

        if np.linalg.det(R) < 0:
            Vt[-1, :] *= -1
            R = Vt.T @ U.T

        frame_pred_aligned = frame_pred_c @ R.T
        error = np.linalg.norm(frame_pred_aligned - frame_target_c)
        procrustes_errors.append(error)

    procrustes_mpjpe = np.mean(procrustes_errors)
    procrustes_std = np.std(procrustes_errors)

    # Velocity error (smoothness metric)
    pred_vel = np.diff(pred_joints, axis=0)
    target_vel = np.diff(target_joints, axis=0)
    vel_error = np.linalg.norm(pred_vel - target_vel, axis=-1)
    mpjve = vel_error.mean()
    mpjve_std = vel_error.std()

    return {
        'per_joint_mpjpe': per_joint_mpjpe,
        'per_joint_std': per_joint_std,
        'global_mpjpe': global_mpjpe,
        'global_std': global_std,
        'procrustes_mpjpe': procrustes_mpjpe,
        'procrustes_std': procrustes_std,
        'mpjve': mpjve,
        'mpjve_std': mpjve_std,
        'joint_count': joint_count,
    }

# Evaluate on test set with timing
print("\nEvaluating on test set...")
import time
eval_start = time.time()
all_preds = []
all_targets = []
with torch.no_grad():
    for batch in tqdm(val_loader):
        input_seq = batch['input'].to(device)
        pose_seq = batch['pose'].to(device)

        pred_pose = model(input_seq)
        all_preds.append(pred_pose.cpu().numpy())
        all_targets.append(pose_seq.cpu().numpy())

eval_time = time.time() - eval_start
all_preds = np.concatenate(all_preds, axis=0)
all_targets = np.concatenate(all_targets, axis=0)
n_frames_eval = all_preds.shape[0] * all_preds.shape[1]
fps = n_frames_eval / eval_time

# EXPLICIT LEARNING: Apply post-processing skeleton rescaling
print("\nApplying EXPLICIT post-processing (skeleton rescaling)...")
all_preds_rescaled = all_preds.copy()
if anthropo_test_raw is not None and len(anthropo_test_raw) > 0:
    # Apply per-subject skeleton rescaling based on anthropometry
    all_preds_rescaled = rescale_skeleton_to_anthropometry(
        all_preds.reshape(-1, all_preds.shape[-1]),
        anthropo_test_raw,
        anthropo_feature_names,
        pose_scaler=preprocessor.pose_scaler,
    ).reshape(all_preds.shape)
    print(f"Skeleton rescaling applied using {len(anthropo_feature_names)} anthropometric features")
else:
    print("No anthropometric data available for post-processing rescaling")

# Compute metrics with original predictions
metrics = compute_detailed_metrics(all_preds, all_targets, scaler=preprocessor.pose_scaler)

# Compute metrics with rescaled predictions (EXPLICIT)
metrics_rescaled = compute_detailed_metrics(all_preds_rescaled, all_targets, scaler=preprocessor.pose_scaler)

print(f"\n{'='*70}")
print(f"Performance Metrics (EXPLICIT LEARNING with post-processing rescaling)")
print(f"{'='*70}")
print(f"Original Predictions:")
print(f"  MPJPE:                {metrics['global_mpjpe']:.4f} ± {metrics['global_std']:.4f}")
print(f"  Procrustes MPJPE:     {metrics['procrustes_mpjpe']:.4f} ± {metrics['procrustes_std']:.4f}")
print(f"  MPJ Velocity Error:   {metrics['mpjve']:.4f} ± {metrics['mpjve_std']:.4f}")
print(f"\nAfter Post-processing Rescaling:")
print(f"  MPJPE:                {metrics_rescaled['global_mpjpe']:.4f} ± {metrics_rescaled['global_std']:.4f}")
print(f"  Procrustes MPJPE:     {metrics_rescaled['procrustes_mpjpe']:.4f} ± {metrics_rescaled['procrustes_std']:.4f}")
print(f"  MPJ Velocity Error:   {metrics_rescaled['mpjve']:.4f} ± {metrics_rescaled['mpjve_std']:.4f}")
print(f"\nInference Speed:        {fps:.2f} frames/sec")
print(f"{'='*70}\n")

# Create visualization with per-joint MPJPE and global metrics using joint names
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

joint_indices = np.arange(metrics['joint_count'])
axes[0, 0].bar(joint_indices, metrics['per_joint_mpjpe'],
               yerr=metrics['per_joint_std'], capsize=4, alpha=0.7, color='steelblue', label='Original')
axes[0, 0].bar(joint_indices + 0.2, metrics_rescaled['per_joint_mpjpe'],
               yerr=metrics_rescaled['per_joint_std'], capsize=4, alpha=0.7, color='coral', label='After Rescaling')
axes[0, 0].set_xlabel('Joint ID', fontsize=11)
axes[0, 0].set_ylabel('MPJPE', fontsize=11)
axes[0, 0].set_title('Per-Joint MPJPE: Original vs Post-processed (EXPLICIT)', fontsize=12, fontweight='bold')
axes[0, 0].set_xticks(joint_indices + 0.1)
axes[0, 0].set_xticklabels(joint_indices, fontsize=9)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3, axis='y')

ax_text = axes[0, 1]
ax_text.axis('off')
metrics_text = f"""Global Performance Metrics (EXPLICIT LEARNING)

ORIGINAL PREDICTIONS:
  MPJPE (Mean):         {metrics['global_mpjpe']:.6f}
  MPJPE (Std Dev):      {metrics['global_std']:.6f}
  Procrustes MPJPE:     {metrics['procrustes_mpjpe']:.6f}
  MPJ Velocity Error:   {metrics['mpjve']:.6f}

AFTER POST-PROCESSING:
  MPJPE (Mean):         {metrics_rescaled['global_mpjpe']:.6f}
  MPJPE (Std Dev):      {metrics_rescaled['global_std']:.6f}
  Procrustes MPJPE:     {metrics_rescaled['procrustes_mpjpe']:.6f}
  MPJ Velocity Error:   {metrics_rescaled['mpjve']:.6f}

INFERENCE SPEED:
  Inference Speed:      {fps:.2f} FPS
  Total Frames Eval:    {n_frames_eval}
  Eval Time:            {eval_time:.2f} sec
"""
ax_text.text(0.1, 0.95, metrics_text, transform=ax_text.transAxes,
             fontsize=9, verticalalignment='top', family='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

per_frame_errors = []
per_frame_errors_rescaled = []
for i in range(metrics['joint_count']):
    per_frame_errors.append([])
    per_frame_errors_rescaled.append([])

all_preds_denorm = preprocessor.pose_scaler.inverse_transform(all_preds.reshape(-1, all_preds.shape[-1]))
all_preds_rescaled_denorm = preprocessor.pose_scaler.inverse_transform(all_preds_rescaled.reshape(-1, all_preds_rescaled.shape[-1]))
all_targets_denorm = preprocessor.pose_scaler.inverse_transform(all_targets.reshape(-1, all_targets.shape[-1]))
all_preds_j = all_preds_denorm.reshape(-1, metrics['joint_count'], 3)
all_preds_rescaled_j = all_preds_rescaled_denorm.reshape(-1, metrics['joint_count'], 3)
all_targets_j = all_targets_denorm.reshape(-1, metrics['joint_count'], 3)

for j in range(metrics['joint_count']):
    errors = np.linalg.norm(all_preds_j[:, j, :] - all_targets_j[:, j, :], axis=1)
    errors_rescaled = np.linalg.norm(all_preds_rescaled_j[:, j, :] - all_targets_j[:, j, :], axis=1)
    per_frame_errors[j] = errors
    per_frame_errors_rescaled[j] = errors_rescaled

# Show comparison between original and rescaled (flattened distributions to keep boxplot input 2D max)
orig_error_dist = np.concatenate(per_frame_errors) if len(per_frame_errors) else np.array([0.0])
rescaled_error_dist = np.concatenate(per_frame_errors_rescaled) if len(per_frame_errors_rescaled) else np.array([0.0])
axes[1, 0].boxplot([orig_error_dist, rescaled_error_dist], labels=['Original', 'Rescaled'])
axes[1, 0].set_ylabel('Error Distribution', fontsize=11)
axes[1, 0].set_title('Error Distribution Comparison: Original vs Rescaled', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

worst_indices = np.argsort(metrics['per_joint_mpjpe'])[-5:][::-1]
worst_joints = [JOINT_NAMES[i] if i < len(JOINT_NAMES) else f"J{i}" for i in worst_indices]
worst_values_orig = metrics['per_joint_mpjpe'][worst_indices]
worst_values_rescaled = metrics_rescaled['per_joint_mpjpe'][worst_indices]

x = np.arange(len(worst_joints))
width = 0.35
axes[1, 1].barh(x - width/2, worst_values_orig, width, label='Original', color='steelblue', alpha=0.7)
axes[1, 1].barh(x + width/2, worst_values_rescaled, width, label='Rescaled', color='coral', alpha=0.7)
axes[1, 1].set_yticks(x)
axes[1, 1].set_yticklabels(worst_joints)
axes[1, 1].set_xlabel('MPJPE', fontsize=11)
axes[1, 1].set_title('Top 5 Worst Joints: Original vs Rescaled (EXPLICIT)', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
inference_fig_path = PREDICTION_RESULTS_DIR / f"{TEST_RESULT_TAG}_inference_results.png"

NameError: name 'PROJECT_ROOT' is not defined

In [ ]:
# Export predictions to CSV for R analysis
print("\nExporting predictions to CSV...")
csv_data = []

# Denormalize all predictions for CSV export
all_preds_denorm = preprocessor.pose_scaler.inverse_transform(all_preds.reshape(-1, all_preds.shape[-1]))
all_targets_denorm = preprocessor.pose_scaler.inverse_transform(all_targets.reshape(-1, all_targets.shape[-1]))

all_preds_j = all_preds_denorm.reshape(-1, metrics['joint_count'], 3)
all_targets_j = all_targets_denorm.reshape(-1, metrics['joint_count'], 3)

# Create long-format CSV with joint names
for frame_id in range(all_preds_j.shape[0]):
    for joint_id in range(metrics['joint_count']):
        gt_x, gt_y, gt_z = all_targets_j[frame_id, joint_id]
        pred_x, pred_y, pred_z = all_preds_j[frame_id, joint_id]
        error = np.linalg.norm(all_preds_j[frame_id, joint_id] - all_targets_j[frame_id, joint_id])
        joint_name = JOINT_NAMES[joint_id] if joint_id < len(JOINT_NAMES) else f"J{joint_id}"

        csv_data.append({
            'frame_id': frame_id,
            'joint_id': joint_id,
            'joint_name': joint_name,
            'gt_x': gt_x,
            'gt_y': gt_y,
            'gt_z': gt_z,
            'pred_x': pred_x,
            'pred_y': pred_y,
            'pred_z': pred_z,
            'error': error,
        })

predictions_df = pd.DataFrame(csv_data)
csv_path = PREDICTION_RESULTS_DIR / f"{TEST_RESULT_TAG}_predictions.csv"
predictions_df.to_csv(str(csv_path), index=False)

print(f"Predictions exported to: {csv_path}")
print(f"  Shape: {predictions_df.shape[0]} rows x {predictions_df.shape[1]} columns")
print(f"  Columns: {', '.join(predictions_df.columns.tolist())}")
print("\nFirst few rows:")
print(predictions_df.head(10))

In [39]:
# 3D HTML animation: Ground Truth vs Prediction over FULL test timeline with fixed global axes
def default_bones_for_joint_count(joint_count):
    if joint_count >= 23:
        return [
            (0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6),
            (4, 7), (4, 11), (7, 8), (8, 9), (9, 10), (11, 12), (12, 13), (13, 14),
            (0, 15), (0, 19), (15, 19),
            (15, 16), (16, 17), (17, 18), (19, 20), (20, 21), (21, 22),
        ]
    return [(i, i + 1) for i in range(max(0, joint_count - 1))]


def predict_full_test_sequence(
    model, pressure_processed, imu_processed, pose_processed, sequence_length, device, segment_ids=None, batch_windows=256
):
    """Predict full timeline by averaging overlapping sliding-window predictions."""
    model.eval()
    n = len(pressure_processed)
    d_out = pose_processed.shape[1]

    # Explicit anthropometric mode: model input is pressure + IMU only.
    input_features = np.concatenate([pressure_processed, imu_processed], axis=1).astype(np.float32)

    pred_sum = np.zeros((n, d_out), dtype=np.float32)
    pred_cnt = np.zeros((n, 1), dtype=np.float32)

    # Build valid starts respecting segments
    starts = []
    max_start = n - sequence_length
    seg = None if segment_ids is None else np.asarray(segment_ids)
    for s in range(max_start + 1):
        if seg is not None and seg[s] != seg[s + sequence_length - 1]:
            continue
        starts.append(s)
    if not starts:
        raise ValueError('No valid windows for full-sequence prediction.')

    with torch.no_grad():
        for i in range(0, len(starts), batch_windows):
            batch_starts = starts[i:i + batch_windows]
            batch_x = np.stack([input_features[s:s + sequence_length] for s in batch_starts], axis=0)
            batch_x_t = torch.from_numpy(batch_x).to(device)
            batch_pred = model(batch_x_t).cpu().numpy()  # (B, T, D)

            for j, s in enumerate(batch_starts):
                pred_sum[s:s + sequence_length] += batch_pred[j]
                pred_cnt[s:s + sequence_length] += 1.0

    pred_cnt[pred_cnt == 0] = 1.0
    pred_avg_std = pred_sum / pred_cnt
    gt_std = pose_processed.copy()

    return gt_std, pred_avg_std


def create_3d_html_animation(
    gt_seq_std, pred_seq_std, pose_scaler, output_html=None,
    step=3, bones=None, swap_yz=True, marker_size=4, frame_duration_ms=60, axis_margin_ratio=0.05,
    title='3D Skeleton Animation: Ground Truth (Red) vs Prediction (Blue)',
):
    gt_seq_std = np.asarray(gt_seq_std)
    pred_seq_std = np.asarray(pred_seq_std)
    if gt_seq_std.shape != pred_seq_std.shape:
        raise ValueError(f'GT and pred shape mismatch: {gt_seq_std.shape} vs {pred_seq_std.shape}')

    t, d = gt_seq_std.shape
    if d % 3 != 0:
        raise ValueError(f'Pose dimension must be divisible by 3, got {d}')
    joint_count = d // 3

    gt_denorm = pose_scaler.inverse_transform(gt_seq_std)
    pred_denorm = pose_scaler.inverse_transform(pred_seq_std)

    gt_xyz = gt_denorm.reshape(t, joint_count, 3)
    pred_xyz = pred_denorm.reshape(t, joint_count, 3)

    both = np.concatenate([gt_xyz, pred_xyz], axis=0)
    mins = both.reshape(-1, 3).min(axis=0)
    maxs = both.reshape(-1, 3).max(axis=0)
    span = np.maximum(maxs - mins, 1e-6)
    margin = span * axis_margin_ratio
    axis_min = mins - margin
    axis_max = maxs + margin

    # Force identical limits on all axes to preserve geometric proportions.
    global_min = float(np.min(axis_min))
    global_max = float(np.max(axis_max))

    if bones is None:
        bones = default_bones_for_joint_count(joint_count)

    def xyz_for_plot(arr):
        if swap_yz:
            return arr[:, 0], arr[:, 2], arr[:, 1]
        return arr[:, 0], arr[:, 1], arr[:, 2]

    def frame_traces(frame_gt, frame_pred):
        traces = []
        xg, yg, zg = xyz_for_plot(frame_gt)
        xp, yp, zp = xyz_for_plot(frame_pred)

        traces.append(go.Scatter3d(
            x=xg, y=yg, z=zg, mode='markers',
            marker=dict(size=marker_size, color='red', opacity=0.85),
            name='Ground Truth Joints', showlegend=True,
        ))
        for a, b in bones:
            if a < len(frame_gt) and b < len(frame_gt):
                traces.append(go.Scatter3d(
                    x=[xg[a], xg[b]], y=[yg[a], yg[b]], z=[zg[a], zg[b]],
                    mode='lines', line=dict(color='red', width=3),
                    name='Ground Truth Bones', showlegend=False,
                ))

        traces.append(go.Scatter3d(
            x=xp, y=yp, z=zp, mode='markers',
            marker=dict(size=marker_size, color='blue', opacity=0.85),
            name='Predicted Joints', showlegend=True,
        ))
        for a, b in bones:
            if a < len(frame_pred) and b < len(frame_pred):
                traces.append(go.Scatter3d(
                    x=[xp[a], xp[b]], y=[yp[a], yp[b]], z=[zp[a], zp[b]],
                    mode='lines', line=dict(color='blue', width=3),
                    name='Predicted Bones', showlegend=False,
                ))
        return traces

    frame_indices = list(range(0, t, max(1, int(step))))
    fig = go.Figure()
    for tr in frame_traces(gt_xyz[frame_indices[0]], pred_xyz[frame_indices[0]]):
        fig.add_trace(tr)

    frames = [
        go.Frame(data=frame_traces(gt_xyz[i], pred_xyz[i]), name=f'frame{k}')
        for k, i in enumerate(frame_indices)
    ]
    fig.frames = frames

    shared_range = [global_min, global_max]
    x_range = shared_range
    y_range = shared_range
    z_range = shared_range

    fig.update_layout(
        title=title, width=1000, height=900, showlegend=True,
        scene=dict(
            xaxis=dict(title='X', range=x_range),
            yaxis=dict(title='Y', range=y_range),
            zaxis=dict(title='Z', range=z_range),
            aspectmode='cube',
        ),
        updatemenus=[{
            'buttons': [
                {'args': [None, {'frame': {'duration': frame_duration_ms, 'redraw': True}, 'fromcurrent': True}], 'label': 'Play', 'method': 'animate'},
                {'args': [[None], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate', 'transition': {'duration': 0}}], 'label': 'Pause', 'method': 'animate'},
            ],
            'direction': 'left', 'pad': {'r': 10, 't': 70}, 'showactive': False,
            'type': 'buttons', 'x': 0.1, 'xanchor': 'right', 'y': 0, 'yanchor': 'top',
        }],
        sliders=[{
            'active': 0, 'yanchor': 'top', 'xanchor': 'left',
            'currentvalue': {'font': {'size': 16}, 'prefix': 'Frame: ', 'visible': True, 'xanchor': 'right'},
            'transition': {'duration': 150, 'easing': 'cubic-in-out'},
            'pad': {'b': 10, 't': 40}, 'len': 0.9, 'x': 0.1, 'y': 0,
            'steps': [
                {'args': [[f.name], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate', 'transition': {'duration': 0}}], 'label': str(i), 'method': 'animate'}
                for i, f in enumerate(frames)
            ],
        }],
    )

    if output_html is None:
        output_html = ANIMATION_RESULTS_DIR / f"{TEST_RESULT_TAG}_gt_vs_pred_full_test.html"
    output_html = Path(output_html)
    output_html.parent.mkdir(parents=True, exist_ok=True)
    output_html.write_text(fig.to_html(full_html=True, include_plotlyjs='cdn'), encoding='utf-8')

    print(f'Animation frames: {len(frames)} (step={step})')
    print(f'Saved 3D animation to: {output_html}')
    return output_html

# Adjustable visualization parameters
VIS_STEP = 3
VIS_OUT_HTML = ANIMATION_RESULTS_DIR / f"{TEST_RESULT_TAG}_gt_vs_pred_full_test.html"
VIS_SWAP_YZ = True
VIS_FRAME_DURATION_MS = 50

# Build full test GT/pred sequences and export animation
full_gt_std, full_pred_std = predict_full_test_sequence(
    model=model,
    pressure_processed=pressure_test_processed,
    imu_processed=imu_test_processed,
    pose_processed=pose_test_processed,
    sequence_length=sequence_length,
    device=device,
    segment_ids=test_segment_ids if 'test_segment_ids' in globals() else None,
    batch_windows=256,
)

create_3d_html_animation(
    gt_seq_std=full_gt_std,
    pred_seq_std=full_pred_std,
    pose_scaler=preprocessor.pose_scaler,
    output_html=VIS_OUT_HTML,
    step=VIS_STEP,
    bones=default_bones_for_joint_count(full_gt_std.shape[1] // 3),
    swap_yz=VIS_SWAP_YZ,
    marker_size=4,
    frame_duration_ms=VIS_FRAME_DURATION_MS,
    axis_margin_ratio=0.05,
    title='GT vs Predicted Skeleton (Fixed Global Axes, Full Test Timeline)',
)

NameError: name 'ANIMATION_RESULTS_DIR' is not defined

## 9. Model Summary and Key Statistics

In [ ]:
print("="*80)
print("SOLEFORMER PIPELINE SUMMARY")
print("="*80)

detected_output_dim = pose_train_processed.shape[1]
detected_num_joints = detected_output_dim // 3 if detected_output_dim % 3 == 0 else 'unknown'

print("\n1. INPUT SPECIFICATION:")
print(f"   - Pressure sensors: {pressure_train_processed.shape[1]} channels total")
print(f"   - IMU data: {imu_train_processed.shape[1]} channels total")
print(f"   - Sequence length: {sequence_length} frames")
print(f"   - Total input features: {pressure_train_processed.shape[1] + imu_train_processed.shape[1]}")

print("\n2. OUTPUT SPECIFICATION:")
print(f"   - Joint count (detected): {detected_num_joints}")
print(f"   - 3D coordinates: X, Y, Z per joint")
print(f"   - Total output features: {detected_output_dim}")

print("\n3. ARCHITECTURE COMPONENTS:")
print(f"   - Pressure feature extractor: GraphPressureNet")
print(f"   - IMU feature extractor: 2-layer MLP")
print(f"   - Transformer d_model: {model_config['d_model']}")
print(f"   - Attention heads: {model_config['nhead']}")
print(f"   - Encoder layers: {model_config['num_encoder_layers']}")
print(f"   - Cross-attention: Bidirectional (pressure ↔ IMU)")
print(f"   - Total SoleFormer parameters: {sum(p.numel() for p in model.parameters()):,}")

print("\n4. LOSS FUNCTION:")
print(f"   - Pose loss (L2): weight=1.0")
print(f"   - IMU cycle loss: weight=0.5")
print(f"   - Pressure cycle loss: weight=0.5")
print(f"   - Total: L = 1.0*L_pose + 0.5*L_imu + 0.5*L_pressure")

print("\n5. TRAINING CONFIGURATION:")
print(f"   - Optimizer: AdamW (lr=1e-3, weight_decay=1e-5)")
print(f"   - Scheduler: CosineAnnealingLR")
print(f"   - Batch size: 32")
print(f"   - Epochs: {train_config['num_epochs']}")
print(f"   - Best validation loss: {best_val_loss:.6f}")

print("\n6. DATASET STATISTICS:")
print(f"   - Train sequences: {len(train_dataset)}")
print(f"   - Test sequences: {len(test_dataset)}")
if 'train_segment_ids' in globals() and train_segment_ids is not None:
    print(f"   - Train segments: {len(np.unique(train_segment_ids))}")
if 'test_segment_ids' in globals() and test_segment_ids is not None:
    print(f"   - Test segments: {len(np.unique(test_segment_ids))}")

print("\n7. EVALUATION METRICS:")
print(f"   - MPJPE (de-normalized): {avg_mpjpe_raw:.4f} original units")
print(f"   - MPJPE (if units are meters): {avg_mpjpe:.2f} mm")

print("\n" + "="*80)

In [ ]:
# Model architecture visualization
print("\nSOLEFORMER ARCHITECTURE:")
print(model)

print("\n\nACCELNET (Auxiliary):")
print(accel_net)

print("\n\nPRESSNET (Auxiliary):")
print(press_net)

## 10. Summary

This notebook implements the complete **SoleFormer** pipeline for 3D human pose estimation from insole sensors:

### Key Components:
1. **Data Preprocessing**: Normalizes pressure (0-20 N/cm² → 0-1) and IMU (standardization)
2. **Two-Stream Architecture**: 
   - Pressure stream uses Graph Neural Network to capture spatial sensor relationships
   - IMU stream uses simple MLP for acceleration/gyro features
3. **Cross-Attention Mechanism**: Learns bidirectional relationships between pressure and IMU
4. **Double-Cycle Consistency Loss**: Enforces physical constraints:
   - IMU cycle: pose → AccelNet → predicted acceleration
   - Pressure cycle: pose → PressNet → predicted pressure
5. **Training**: End-to-end supervised learning with sequence-to-sequence mapping

### Advantages over Prior Work:
- Minimal hardware (1 pair of insoles vs 2-6 IMUs for other methods)
- Real-time performance (~11ms inference)
- Accurate pose estimation (65-70 mm MPJPE vs 70+ mm for other wearable methods)
- Physical constraints improve generalization
- Works in diverse activities (skiing, walking, etc.)

### References:
Wu, E., Khirodkar, R., Koike, H., & Kitani, K. (2024). SolePoser: Real-Time 3D Human Pose Estimation using Insole Pressure Sensors. In UIST '24.